In [8]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install -U nltk

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 1: Imports
import pandas as pd
import numpy as np # For metrics later
import torch
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support 
import re
# Hugging Face Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd # For display if needed
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# For sentence tokenization (NLTK is generally good)
import nltk
nltk.download('punkt_tab')
try:
    from nltk.tokenize import sent_tokenize
except ImportError:
    print("NLTK not found. Please install it ('pip install nltk') and run the prerequisite cell to download 'punkt'.")
    print("Falling back to a very basic sentence splitter (less accurate).")

print("Cell 1 executed: Libraries imported.")

/home/raham/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cell 1 executed: Libraries imported.


[nltk_data] Downloading package punkt_tab to /home/raham/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
pip install spacy==3.7.2


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
!python -m spacy download en_core_web_sm

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [9]:
# Cell 2 (Revised): Load CSV Data & Initialize Tokenizer

import pandas as pd
from transformers import AutoTokenizer

# 1. Load your CSV Data
# !!! REPLACE 'your_file_path.csv' WITH THE ACTUAL PATH TO YOUR CSV FILE !!!
try:
    df_full = pd.read_csv('annotated_corpus_for_dataverse.csv')
    print("Successfully loaded data from CSV.")
    # For now, to make the code runnable without your actual file,
    # I'll create a DataFrame that matches your structure with a few examples.
    # WHEN YOU RUN THIS, UNCOMMENT THE LINE ABOVE AND COMMENT OUT THE SAMPLE BELOW.

    # Select relevant columns and create the binary 'label'
    # Assuming 'text_segmen' is your text column and 'relevance_label' is your label column
    # You might need to adjust column names if they are slightly different in your actual CSV
    df = pd.DataFrame()
    df['text_segment'] = df_full['text_segment'] # Make sure 'text_segmen' is correct

    # Convert 'relevance_label' to binary label: 0 if relevance_label is 0, else 1
    df['label'] = df_full['relevance_label'].apply(lambda x: 0 if x == 0 else 1)

    print("\nProcessed DataFrame head with 'text_segment' and 'label':")
    print(df.head())
    print(f"\nValue counts for 'label' column:\n{df['label'].value_counts()}")

except FileNotFoundError:
    print("ERROR: CSV file not found. Please update 'your_file_path.csv'.")
    df = pd.DataFrame() # Create empty df to avoid further errors if file not found
except KeyError as e:
    print(f"ERROR: A column was not found in the CSV: {e}. Please check column names.")
    print("Expected 'text_segmen' and 'relevance_label'.")
    df = pd.DataFrame() # Create empty df
except Exception as e:
    print(f"An unexpected error occurred while loading or processing the CSV: {e}")
    df = pd.DataFrame()

# 2. Define the pre-trained model name
# MODEL_NAME = "distilbert-base-uncased" # For a quicker start
MODEL_NAME = "facebook/bart-base" # As per your interest in BART

# 3. Load the tokenizer
if not df.empty: # Only proceed if DataFrame was loaded/created successfully
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        print(f"\nTokenizer for '{MODEL_NAME}' loaded successfully.")
    except Exception as e:
        print(f"\nError loading tokenizer for '{MODEL_NAME}': {e}")
        print("Please ensure you have an internet connection and the model name is correct.")
        print("If you haven't run `pip install transformers torch`, please do so.")
else:
    print("\nSkipping tokenizer loading as DataFrame is empty due to previous errors.")

Successfully loaded data from CSV.

Processed DataFrame head with 'text_segment' and 'label':
                                        text_segment  label
0  #text': '(Lambin et al., 2003)'}], '#text': 'W...      0
1  #text': 'After the droughts in the 1970s and 1...      1
2  #text': 'Few investment opportunities are avai...      0
3  #text': 'Pastoral production has often existed...      1
4  #text': 'The forests of West and Central Afric...      1

Value counts for 'label' column:
label
0    570
1    233
Name: count, dtype: int64

Tokenizer for 'facebook/bart-base' loaded successfully.


In [10]:

try:
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text_segment'].tolist(),  # Convert to list for tokenization
        df['label'].tolist(),        # Convert to list
        test_size=0.1,               # 20% for validation
        random_state=42,             # For reproducibility
        stratify=df['label'].tolist() # Stratify by labels
    )
    print(f"Data split: {len(train_texts)} training samples, {len(val_texts)} validation samples.")
    # Check stratification (optional, but good for sanity)
    from collections import Counter
    print(f"Train label distribution: {Counter(train_labels)}")
    print(f"Validation label distribution: {Counter(val_labels)}")

except Exception as e:
    print(f"Error during data splitting: {e}")
    # If splitting fails, create empty lists to avoid downstream errors, though training won't work
    train_texts, val_texts, train_labels, val_labels = [], [], [], []


# 2. Tokenize the text data
# We need to handle the case where the tokenizer might not have been loaded due to earlier errors
if 'tokenizer' in globals() and tokenizer is not None and train_texts:

    MAX_LENGTH = 1024 #

    train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
    val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
    print(f"\nTokenization complete. Example train_encodings keys: {list(train_encodings.keys())}")
    # print(f"First training sample input_ids: {train_encodings['input_ids'][0][:20]}...") # See first 20 tokens
else:
    print("\nSkipping tokenization as tokenizer is not available or no training data.")
    train_encodings, val_encodings = None, None # Ensure they are defined for robustness


# 3. Create a PyTorch Dataset class
class LULCDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # For item, convert each value in encodings dict to a tensor
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]) # 'labels' is the expected key for the loss function
        return item

    def __len__(self):
        # The length is the number of samples, assuming all encoding lists have the same length
        # (which they should after tokenizer(..., padding=True))
        return len(self.labels)

# Create dataset objects, only if encodings were successful
if train_encodings and val_encodings:
    train_dataset = LULCDataset(train_encodings, train_labels)
    val_dataset = LULCDataset(val_encodings, val_labels)
    print(f"\nPyTorch Datasets created. Training dataset length: {len(train_dataset)}, Validation dataset length: {len(val_dataset)}")
    # Let's inspect the first item from the train_dataset to see its structure
    if len(train_dataset) > 0:
        print(f"First item from train_dataset: {train_dataset[0]}")
else:
    print("\nSkipping PyTorch Dataset creation as tokenized encodings are not available.")

Data split: 722 training samples, 81 validation samples.
Train label distribution: Counter({0: 513, 1: 209})
Validation label distribution: Counter({0: 57, 1: 24})

Tokenization complete. Example train_encodings keys: ['input_ids', 'attention_mask']

PyTorch Datasets created. Training dataset length: 722, Validation dataset length: 81
First item from train_dataset: {'input_ids': tensor([   0, 9344, 1022,  ...,    1,    1,    1]), 'attention_mask': tensor([1, 1, 1,  ..., 0, 0, 0]), 'labels': tensor(0)}


In [11]:
# Cell: Analyze Token Lengths and Count Sentences > 512 Tokens

# Ensure 'df' and 'tokenizer' are available from previous cells
if 'df' in globals() and not df.empty and 'tokenizer' in globals() and tokenizer is not None:
    all_texts = df['text_segment'].tolist()
    token_lengths = []
    sentences_longer_than_512 = 0
    limit = 1024

    print(f"Analyzing token lengths for {len(all_texts)} sentences...\n")

    for i, text in enumerate(all_texts):
        # Using encode gets the token IDs directly, len() gives the token count
        # add_special_tokens=True includes tokens like <s> and </s> in the count
        tokens = tokenizer.encode(text, add_special_tokens=True)
        current_length = len(tokens)
        token_lengths.append(current_length)

        if current_length > limit:
            sentences_longer_than_512 += 1
            # Optional: Print the text of very long sentences for inspection
            # print(f"Sentence {i+1} is longer than {limit} tokens (length: {current_length}):\n{text[:300]}...\n---")

    import numpy as np # Ensure numpy is imported
    print(f"\n--- Token Length Statistics ---")
    if token_lengths: # Check if token_lengths is not empty
        print(f"Total sentences analyzed: {len(token_lengths)}")
        print(f"Min token length: {np.min(token_lengths)}")
        print(f"Max token length: {np.max(token_lengths)}")
        print(f"Mean token length: {np.mean(token_lengths):.2f}")
        print(f"Median token length: {np.median(token_lengths)}")
        print(f"95th percentile token length: {np.percentile(token_lengths, 95):.2f}")
        print(f"99th percentile token length: {np.percentile(token_lengths, 99):.2f}")
        print(f"\nNumber of sentences with token length > {limit}: {sentences_longer_than_512}")
        percentage_longer = (sentences_longer_than_512 / len(token_lengths)) * 100 if len(token_lengths) > 0 else 0
        print(f"Percentage of sentences with token length > {limit}: {percentage_longer:.2f}%")
    else:
        print("No sentences were processed for token length analysis.")

else:
    print("DataFrame 'df' or 'tokenizer' not found or df is empty. Please ensure previous cells executed successfully.")

Analyzing token lengths for 803 sentences...


--- Token Length Statistics ---
Total sentences analyzed: 803
Min token length: 6
Max token length: 1928
Mean token length: 63.75
Median token length: 36.0
95th percentile token length: 185.50
99th percentile token length: 707.34

Number of sentences with token length > 1024: 6
Percentage of sentences with token length > 1024: 0.75%


In [12]:
import torch

print(f"PyTorch version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
    # Get CUDA version PyTorch was compiled with
    try:
        # For newer PyTorch versions
        from torch.version import cuda as pytorch_cuda_version
        print(f"PyTorch built with CUDA version: {pytorch_cuda_version}")
    except ImportError:
        # For older PyTorch, this might be harder to get directly or might be in torch.version.cuda
        print("Could not directly determine PyTorch's CUDA build version (try torch.version.cuda if available).")
else:
    print("CUDA is not available to PyTorch.")
    print("If you have an NVIDIA GPU and drivers, your PyTorch installation might be CPU-only.")
    print("You may need to reinstall PyTorch with CUDA support specific to your system's CUDA version.")
    print("Visit: https://pytorch.org/get-started/locally/")

PyTorch version: 2.4.1+cu118
CUDA available: True
Number of GPUs: 1
Current CUDA device: 0
Device name: GRID V100S-32Q
PyTorch built with CUDA version: 11.8


In [13]:
# Cell 4b (Corrected Indentation): Initialize Model, Training Arguments, and Trainer

import transformers
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np


# Ensure MODEL_NAME, train_dataset, and val_dataset are available
if 'MODEL_NAME' not in globals() or 'train_dataset' not in globals() or 'val_dataset' not in globals():
    print("ERROR: Essential variables not found. Please run previous cells.")
else:
    print(f"Using MODEL_NAME: {MODEL_NAME}")
    print(f"Transformers library version: {transformers.__version__}")
    print(f"Train dataset length: {len(train_dataset) if train_dataset else 'Not available'}")
    print(f"Validation dataset length: {len(val_dataset) if val_dataset else 'Not available'}")

    # 1. Load the pre-trained model
    try:
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        print(f"\nModel '{MODEL_NAME}' loaded successfully with num_labels=2.")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        print(f"Model moved to device: {device}")
    except Exception as e:
        print(f"Error loading model: {e}")
        model = None

    if model and train_dataset and len(train_dataset) > 0:

        # 2. Define a Compute Metrics function
        def compute_metrics(pred):
            print("\n--- Inside compute_metrics ---") # Debug print
            print(f"Type of pred.predictions: {type(pred.predictions)}")
            print(f"Shape/Len of pred.predictions: {pred.predictions.shape if hasattr(pred.predictions, 'shape') else len(pred.predictions) if hasattr(pred.predictions, '__len__') else 'N/A'}")

            if isinstance(pred.predictions, list) and len(pred.predictions) > 0:
                print(f"Type of first element in pred.predictions: {type(pred.predictions[0])}")
                print(f"Shape/Len of first element: {pred.predictions[0].shape if hasattr(pred.predictions[0], 'shape') else len(pred.predictions[0]) if hasattr(pred.predictions[0], '__len__') else 'N/A'}")
                if len(pred.predictions) > 1:
                    print(f"Type of second element in pred.predictions: {type(pred.predictions[1])}")
                    print(f"Shape/Len of second element: {pred.predictions[1].shape if hasattr(pred.predictions[1], 'shape') else len(pred.predictions[1]) if hasattr(pred.predictions[1], '__len__') else 'N/A'}")

            actual_predictions_for_argmax = None
            if isinstance(pred.predictions, tuple) and len(pred.predictions) > 0 and isinstance(pred.predictions[0], np.ndarray):
                print("pred.predictions is a tuple, trying its first element for argmax.")
                actual_predictions_for_argmax = pred.predictions[0]
            elif isinstance(pred.predictions, np.ndarray) and pred.predictions.ndim == 2:
                print("pred.predictions is already a 2D numpy array.")
                actual_predictions_for_argmax = pred.predictions
            else:
                print("pred.predictions is not a 2D numpy array or expected tuple. This will likely fail or needs careful handling.")
                try:
                    print("Attempting to stack pred.predictions assuming it's a list of logit arrays...")
                    actual_predictions_for_argmax = np.vstack(pred.predictions)
                    print(f"Shape after vstack: {actual_predictions_for_argmax.shape}")
                except Exception as e_stack:
                    print(f"Could not np.vstack pred.predictions: {e_stack}")
                    return {'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0}

            if actual_predictions_for_argmax is not None:
                try:
                    preds = np.argmax(actual_predictions_for_argmax, axis=1)
                    labels = pred.label_ids
                    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
                    acc = accuracy_score(labels, preds)
                    print("--- Exiting compute_metrics (success) ---")
                    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}
                except Exception as e_argmax:
                    print(f"Error during argmax or metric calculation: {e_argmax}")
                    print(f"Shape of actual_predictions_for_argmax: {actual_predictions_for_argmax.shape if hasattr(actual_predictions_for_argmax, 'shape') else 'N/A'}")
                    return {'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
            else:
                print("actual_predictions_for_argmax is None. Returning dummy metrics.")
                return {'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0}

        # --- End of compute_metrics definition ---

        # 3. Define Training Arguments (Correct Indentation)
        OUTPUT_DIR = 'LULC_BART_model_results' # Make sure these are defined at this scope if not globals
        NUM_TRAIN_EPOCHS = 15
        PER_DEVICE_TRAIN_BATCH_SIZE = 8

        steps_per_epoch = 100 # Default fallback
        if PER_DEVICE_TRAIN_BATCH_SIZE > 0 and train_dataset and len(train_dataset) > 0: # ensure train_dataset is checked here too
            steps_per_epoch = math.ceil(len(train_dataset) / PER_DEVICE_TRAIN_BATCH_SIZE)
            if NUM_TRAIN_EPOCHS > 0 and steps_per_epoch == 0:
                steps_per_epoch = 1
        print(f"Calculated steps per epoch: {steps_per_epoch}")

        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=8,
            warmup_steps=50,
            weight_decay=0.01,
            logging_dir='./LULC_BART_logs',
            do_eval=True,
            eval_steps=steps_per_epoch if steps_per_epoch > 0 else 500,
            save_steps=steps_per_epoch if steps_per_epoch > 0 else 500,
            logging_steps=50,
            load_best_model_at_end=False,
            save_total_limit=2,
        )
        print(f"\nTrainingArguments defined with load_best_model_at_end=False. Output directory: {OUTPUT_DIR}")
        print(f"Effective eval_steps: {training_args.eval_steps if hasattr(training_args, 'eval_steps') else 'N/A'}")
        print(f"Effective save_steps: {training_args.save_steps if hasattr(training_args, 'save_steps') else 'N/A'}")

        # 4. Initialize the Trainer (Correct Indentation)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics # This uses the function defined above
        )
        print("\nTrainer initialized.")

    elif not model:
        print("\nSkipping TrainingArguments and Trainer initialization as model was not loaded.")
    else: # train_dataset must be None or (exists and is empty)
        print("\nSkipping TrainingArguments and Trainer initialization as train_dataset is not available or empty.")

Using MODEL_NAME: facebook/bart-base
Transformers library version: 4.46.3
Train dataset length: 722
Validation dataset length: 81


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model 'facebook/bart-base' loaded successfully with num_labels=2.
Model moved to device: cuda
Calculated steps per epoch: 91


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>={ACCELERATE_MIN_VERSION}'`

In [ ]:
# Cell 5: Start Training

# Ensure the 'trainer' object exists from Cell 4
if 'trainer' in globals() and trainer is not None:
    print("Starting model training...")
    try:
        train_result = trainer.train()
        print("\nTraining finished.")

        # Save the final model and tokenizer.
        # Since load_best_model_at_end=False, this saves the model from the last epoch.
        final_model_path = "./LULC_BART_final_model"
        trainer.save_model(final_model_path)

        # If tokenizer was loaded in a previous cell and you want to save it alongside:
        if 'tokenizer' in globals() and tokenizer is not None:
             tokenizer.save_pretrained(final_model_path)
             print(f"Final model and tokenizer saved to {final_model_path}")
        else:
            print(f"Final model saved to {final_model_path}. Tokenizer not found in global scope to save.")


        # You can also log metrics from the training result
        if hasattr(train_result, 'metrics'):
            print("\nTraining Metrics (from train_result):")
            for key, value in train_result.metrics.items():
                print(f"  {key}: {value}")

        # The trainer object itself also has a log_history attribute that might contain more details
        if trainer.state.log_history:
            print("\nFull Log History (from trainer.state.log_history):")
            for log_entry in trainer.state.log_history:
                print(f"  {log_entry}")


    except Exception as e:
        print(f"\nAn error occurred during training: {e}")
        import traceback
        traceback.print_exc() # Print detailed traceback for debugging
else:
    print("Trainer object not found. Please ensure Cell 4 executed successfully and 'trainer' was initialized.")

In [ ]:
# Cell 6: Explicit Evaluation and Making Predictions (Full Modified Version)

from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer
import torch
import numpy as np
import pandas as pd # For creating a DataFrame for predictions

# --- Configuration ---
# Path to the model you want to load for evaluation/prediction
MODEL_PATH = "LULC_BART_final_model"
# Or if you know a specific checkpoint:
# MODEL_PATH = "LULC_BART_model_results/checkpoint-XXX"

# --- Ensure necessary variables are available ---
val_dataset_exists = 'val_dataset' in globals() and val_dataset is not None
global_tokenizer_exists = 'tokenizer' in globals() and tokenizer is not None
global_compute_metrics_exists = 'compute_metrics' in globals() and compute_metrics is not None # from Cell 4
original_trainer_exists = 'trainer' in globals() and trainer is not None # trainer object from Cell 4

# Attempt to reload tokenizer if not globally available
if not global_tokenizer_exists:
    try:
        print(f"Global tokenizer not found. Attempting to load tokenizer from {MODEL_PATH}...")
        # This redefines the global 'tokenizer' if successful
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        global_tokenizer_exists = True # Update flag
        print("Tokenizer reloaded successfully from model path.")
    except Exception as e:
        print(f"ERROR: Global 'tokenizer' not found and could not reload from {MODEL_PATH}: {e}")
        tokenizer = None # Ensure it's None if failed

# --- 0. Inspect val_dataset (Debugging Step) ---
if val_dataset_exists and len(val_dataset) > 0:
    print("\n--- Inspecting first item of val_dataset: ---")
    sample_item = val_dataset[0] # Assuming val_dataset is a PyTorch Dataset
    for key, value in sample_item.items():
        print(f"Key: {key}, Type: {type(value)}, Shape: {value.shape if hasattr(value, 'shape') else 'N/A'}")
    print("--- End of inspection ---")
elif not val_dataset_exists:
    print("WARNING: 'val_dataset' not found. Cannot inspect or perform explicit evaluation later.")
else: # val_dataset exists but is empty
    print("WARNING: 'val_dataset' is empty. Cannot inspect or perform explicit evaluation.")


# --- 1. Load the Fine-Tuned Model ---
loaded_model = None
if global_tokenizer_exists: # Proceed only if tokenizer is available
    try:
        print(f"\nLoading fine-tuned model from: {MODEL_PATH}")
        loaded_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        loaded_model.to(device)
        loaded_model.eval() # Set the model to evaluation mode
        print(f"Model loaded successfully from {MODEL_PATH} and set to evaluation mode on {device}.")
    except Exception as e:
        print(f"Error loading model from {MODEL_PATH}: {e}")
else:
    print("\nSkipping model loading as tokenizer is not available.")


# --- 2. Explicit Evaluation on the Validation Set ---
if loaded_model and val_dataset_exists and len(val_dataset) > 0 and global_tokenizer_exists and global_compute_metrics_exists:
    print("\n--- Performing Explicit Evaluation on Validation Set ---")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Use args from the original trainer if available, otherwise create minimal default ones for evaluation.
    eval_args = None
    if original_trainer_exists:
        eval_args = trainer.args
        # Modify batch size if needed for eval, though original trainer.args should have per_device_eval_batch_size
        # eval_args.per_device_eval_batch_size = 16 # Example
    else:
        # Create minimal TrainingArguments if original trainer/args are not available
        # This part might need adjustment based on your transformers version if original_trainer is gone
        from transformers import TrainingArguments # Import if not globally available from Cell 4 anymore
        print("Original trainer/args not found, creating minimal TrainingArguments for evaluation.")
        temp_output_dir = "./temp_eval_results" # Temporary for this eval trainer
        # Ensure required arguments for TrainingArguments for your transformers version are met
        # For v4.5.1, output_dir is essential. Other args have defaults.
        try:
            eval_args = TrainingArguments(
                output_dir=temp_output_dir,
                per_device_eval_batch_size=16, # Set a reasonable default
                # no_cuda=(device.type == 'cpu'), # Older argument
                # Remove other training-specific args if they cause issues
            )
        except TypeError as te: # Catch issues if TrainingArguments init fails (e.g. missing required args for specific version)
             print(f"Error creating minimal TrainingArguments: {te}. Falling back to very basic.")
             eval_args = TrainingArguments(output_dir=temp_output_dir) # Absolute minimal


    if eval_args is not None:
        temp_trainer_for_eval = Trainer(
            model=loaded_model,
            args=eval_args,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics, # From global scope (defined in Cell 4)
            data_collator=data_collator
        )

        print("Evaluating using the reloaded model and DataCollatorWithPadding...")
        try:
            eval_results = temp_trainer_for_eval.evaluate()
            print("\nValidation Set Evaluation Results:")
            for key, value in eval_results.items():
                print(f"  {key}: {value:.4f}")
        except Exception as e_eval:
            print(f"Error during temp_trainer_for_eval.evaluate(): {e_eval}")
            import traceback
            traceback.print_exc()
    else:
        print("Could not obtain or create TrainingArguments for evaluation.")

else:
    print("\nSkipping explicit evaluation due to missing components (model, val_dataset, tokenizer, or compute_metrics).")
    if not loaded_model: print("- Model not loaded.")
    if not val_dataset_exists or (val_dataset_exists and len(val_dataset) == 0): print("- Val_dataset not available or empty.")
    if not global_tokenizer_exists: print("- Tokenizer not available.")
    if not global_compute_metrics_exists: print("- Compute_metrics function not available.")


# --- 3. Making Predictions on New, Example Sentences ---
if loaded_model and global_tokenizer_exists:
    print("\n--- Making Predictions on New Sentences ---")
    new_sentences = [
        "The expansion of palm oil plantations is causing deforestation in Southeast Asia.",
        "estimate is thus a metric of deforestation occurring in landscapes where agriculture is the dominant direct driver of forest loss rather than only deforestation resulting in agricultural production per se . ",
        "The local library announced a new summer reading program for children.",
        "This could be due to land cover and land use change.",
        "This new AI technique is groundbreaking for image recognition.",
        "Shifting cultivation practices impact forest cover over time."
    ]

    PRED_MAX_LENGTH = 1024 # Ensure this matches your training MAX_LENGTH

    inputs = tokenizer(
        new_sentences,
        padding=True,
        truncation=True,
        max_length=PRED_MAX_LENGTH,
        return_tensors="pt"
    )

    inputs = {k: v.to(loaded_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

    results_df = pd.DataFrame({
        'Sentence': new_sentences,
        'Predicted_Label': predictions.cpu().numpy(),
        'Probability_Class_0 (Not_LULC)': probabilities[:, 0].cpu().numpy(),
        'Probability_Class_1 (LULC)': probabilities[:, 1].cpu().numpy()
    })
    print(results_df)
    print("\nInterpretation: 'Predicted_Label' 1 usually means LULC-related, 0 means not.")
else:
    print("\nSkipping new predictions as model or tokenizer is not available.")
    if not loaded_model: print("- Model not loaded for prediction.")
    if not global_tokenizer_exists: print("- Tokenizer not available for prediction.")

In [3]:
# Cell 7d (Corrected NLTK Exception Handling)

# 1. Install NLTK if not already installed
try:
    import nltk
    print("NLTK is already installed.")
except ModuleNotFoundError:
    print("NLTK not found. Installing NLTK...")
    import sys
    # In a Jupyter/Colab environment, use !pip install
    process_output = !{sys.executable} -m pip install nltk
    # print("\n".join(process_output)) # Optional: Print output of pip install
    import nltk # Try importing again after installation
    print("NLTK installed successfully.")
except Exception as e_install: # Catch any other import/install error
    print(f"An error occurred during NLTK installation or import: {e_install}")
    nltk = None # Ensure nltk is None if import failed

# 2. Download the 'punkt' tokenizer models, only if nltk was successfully imported/installed
if nltk:
    try:
        nltk.data.find('tokenizers/punkt') # This line raises LookupError if 'punkt' isn't found
        print("NLTK 'punkt' resource is already available.")
    except LookupError: # Corrected exception type
        print("NLTK 'punkt' resource not found. Downloading...")
        try:
            nltk.download('punkt')
            print("NLTK 'punkt' resource downloaded successfully.")
        except Exception as e_download: # Catch potential errors during download
            print(f"Error downloading 'punkt': {e_download}")
    except Exception as e_find: # Catch other potential errors with nltk.data.find
        print(f"An error occurred checking for NLTK 'punkt' resource: {e_find}")
else:
    print("NLTK was not successfully imported or installed. Cannot download 'punkt'.")


print("\nNLTK setup check complete for sentence tokenization.")

NLTK is already installed.
NLTK 'punkt' resource is already available.

NLTK setup check complete for sentence tokenization.


In [4]:
# Cell 7d (Revised - NLTK Installation and Resource Download)

# 1. Install NLTK if not already installed
try:
    import nltk
    print("NLTK is already installed.")
except ModuleNotFoundError:
    print("NLTK not found. Installing NLTK...")
    import sys
    # In a Jupyter/Colab environment, use !pip install
    # If running a standalone .py script, you'd typically run 'pip install nltk' in your terminal first.
    # For this notebook context:
    process_output = !{sys.executable} -m pip install nltk
    # print("\n".join(process_output)) # Print output of pip install
    import nltk # Try importing again after installation
    print("NLTK installed successfully.")

# 2. Download the 'punkt' tokenizer models
try:
    nltk.data.find('tokenizers/punkt')
    print("NLTK 'punkt' resource is already available.")
except nltk.downloader.DownloadError:
    print("NLTK 'punkt' resource not found. Downloading...")
    nltk.download('punkt')
    print("NLTK 'punkt' resource downloaded successfully.")
except Exception as e:
    print(f"An error occurred with NLTK 'punkt' resource: {e}")

print("\nNLTK setup complete for sentence tokenization.")

NLTK is already installed.
NLTK 'punkt' resource is already available.

NLTK setup complete for sentence tokenization.


In [5]:
!pip install -U ipywidgets

Requirement already up-to-date: ipywidgets in /home/raham/venv/lib/python3.8/site-packages (8.1.7)


In [6]:
# Cell 8: Extract LULC-Related Sentences from an Article

import pandas as pd # For display if needed
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# For sentence tokenization (NLTK is generally good)
import nltk
nltk.download('punkt_tab')
try:
    from nltk.tokenize import sent_tokenize
except ImportError:
    print("NLTK not found. Please install it ('pip install nltk') and run the prerequisite cell to download 'punkt'.")
    print("Falling back to a very basic sentence splitter (less accurate).")
    import re
    def sent_tokenize_basic(text): # Basic fallback
        # Simple split by common sentence-ending punctuation followed by space or end of string.
        # This is very rudimentary and won't handle all cases well (e.g., Mr. Smith).
        sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|!)\s', text)
        return [s.strip() for s in sentences if s.strip()]
    sent_tokenize = sent_tokenize_basic


# --- 1. Define Your Article Text ---
# Replace the example text below with your actual article content.
# For long articles, consider reading from a file:
# with open("your_article.txt", "r", encoding="utf-8") as f:
#     article_text = f.read()

article_text = """
Rapid urbanization is a defining feature of the 21st century, particularly in developing nations.
This trend often leads to significant changes in land use and land cover (LULC). For instance,
agricultural land is frequently converted to residential and commercial areas. The local weather has been quite pleasant this week.
Such transformations can strain existing infrastructure and natural resources.
Deforestation for new settlements or to clear land for farming also contributes to LULC dynamics.
These changes have profound implications for biodiversity, water cycles, and local climates.
Many economists are discussing the latest inflation figures. Sustainable land management practices
are crucial to mitigate the negative impacts of LULC change and promote resilient development.
The upcoming festival is expected to draw large crowds to the city center. Furthermore, climate change
itself can exacerbate LULC issues, such as increased desertification or coastal erosion.
"""

print("--- Article Text Provided ---")


# --- 2. Sentence Segmentation ---
try:
    article_sentences = sent_tokenize(article_text)
    print(f"\n--- Segmented Article into {len(article_sentences)} Sentences ---")
    # for i, sentence in enumerate(article_sentences):
    #     print(f"{i+1}: {sentence}")
except Exception as e:
    print(f"Error during sentence tokenization: {e}")
    article_sentences = []


# --- 3. Load Model and Tokenizer (if not already in scope or to ensure fresh load) ---
MODEL_PATH_FOR_ARTICLE = "./LULC_BART_final_model" # Path to your saved fine-tuned model
PRED_MAX_LENGTH_FOR_ARTICLE = 1024 # Should match the max_length used for training

# Check if 'loaded_model' and 'tokenizer' from previous cells are still valid and what you want to use.
# For this specific task, it's often safer to reload to ensure you use the correct saved artifact.
article_model = None
article_tokenizer = None

try:
    print(f"\nLoading model and tokenizer from {MODEL_PATH_FOR_ARTICLE} for article processing...")
    article_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH_FOR_ARTICLE)
    article_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH_FOR_ARTICLE)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    article_model.to(device)
    article_model.eval() # Set to evaluation mode
    print("Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model/tokenizer: {e}. Cannot proceed with classification.")


# --- 4. Classify Each Sentence ---
lulc_related_sentences = []
if article_model and article_tokenizer and article_sentences:
    print("\n--- Classifying Sentences ---")
    
    # We can process sentences in batches for efficiency, but for simplicity here, one by one.
    # For larger numbers of sentences, batching is recommended (as in the prediction part of Cell 6)

    for sentence in article_sentences:
        if not sentence.strip(): # Skip empty sentences
            continue

        inputs = article_tokenizer(
            sentence,
            padding=True,        # Padding to max_length or longest in batch (if batching)
            truncation=True,     # Truncate to max_length
            max_length=PRED_MAX_LENGTH_FOR_ARTICLE,
            return_tensors="pt"  # Return PyTorch tensors
        )
        inputs = {k: v.to(article_model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = article_model(**inputs)
            logits = outputs.logits
            # probabilities = torch.softmax(logits, dim=-1) # Optional if you want confidence scores
            prediction = torch.argmax(logits, dim=-1).item() # .item() to get Python number

        if prediction == 1: # Assuming 1 is your "LULC related" label
            lulc_related_sentences.append(sentence)
            # print(f"  [LULC Related]: {sentence}") # Optional: print as it finds them
        # else:
            # print(f"  [Not Related]: {sentence}")

    print(f"\nFinished classification. Found {len(lulc_related_sentences)} LULC-related sentences.")
else:
    if not (article_model and article_tokenizer):
        print("Skipping classification as model or tokenizer was not loaded.")
    if not article_sentences:
        print("Skipping classification as no sentences were extracted from the article.")


# --- 5. Display Extracted LULC-Related Sentences ---
if lulc_related_sentences:
    print("\n--- Extracted LULC-Related Sentences: ---")
    for i, sentence in enumerate(lulc_related_sentences):
        print(f"{i+1}. {sentence}")
else:
    print("\nNo LULC-related sentences were identified in the article.")

--- Article Text Provided ---

--- Segmented Article into 11 Sentences ---

Loading model and tokenizer from ./LULC_BART_final_model for article processing...
Error loading model/tokenizer: Incorrect path_or_model_id: './LULC_BART_final_model'. Please provide either the path to a local folder or the repo_id of a model on the Hub.. Cannot proceed with classification.
Skipping classification as model or tokenizer was not loaded.

No LULC-related sentences were identified in the article.


[nltk_data] Downloading package punkt_tab to /home/raham/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
# Cell 9: Process Multiple Articles from CSV to Extract LULC Sentences (Corrected Preprocessing Logic)

CSV_FILE_PATH = 'extracted_data_full.csv'
# !!! REPLACE these with the actual names of your text columns !!!
TEXT_COLUMNS_TO_CONCATENATE = ['title', 'abstract', 'sections'] # Example names

ARTICLE_ID_COLUMN = None   # Example name

MODEL_PATH_FOR_ARTICLES_CSV = "LULC_BART_final_model"
PRED_MAX_LENGTH_FOR_ARTICLES_CSV = 1024

# --- Preprocessing Function Definition ---
def preprocess_article_text(text):
    if not isinstance(text, str): # Handle non-string inputs gracefully
        return ""
        
    # 1. Basic whitespace normalization
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Remove [1], [23] style citations
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text) # Handles [1], [1,2], [1, 2, 3]
    
    # 3. Attempt to remove (Author, YYYY) or (Author et al., YYYY)
    # This regex is simple and might need refinement for complex cases
    text = re.sub(r"\(\s*[\w\s\-\.']+(?:et\s+al\.?)?,\s*\d{4}\s*\)", '', text) 
    
    # 4. Example: Remove URLs (add this if URLs are common in your text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 5. ADD OTHER PREPROCESSING STEPS HERE AS NEEDED
    # (e.g., specific pattern removal for '#text': ..., HTML cleaning if source is web)
    
    return text

# --- 1. Load Articles from CSV ---
try:
    print(f"Loading articles from CSV: {CSV_FILE_PATH}")
    articles_df = pd.read_csv(CSV_FILE_PATH)
    print(f"Successfully loaded {len(articles_df)} articles.")
    missing_text_cols = [col for col in TEXT_COLUMNS_TO_CONCATENATE if col not in articles_df.columns]
    if missing_text_cols:
        raise ValueError(f"Missing text columns in CSV: {missing_text_cols}. Expected: {TEXT_COLUMNS_TO_CONCATENATE}")
    if ARTICLE_ID_COLUMN and ARTICLE_ID_COLUMN not in articles_df.columns:
        print(f"Warning: Article ID column '{ARTICLE_ID_COLUMN}' not found. Using row index for article identification.")
except FileNotFoundError:
    print(f"ERROR: CSV file not found at {CSV_FILE_PATH}.")
    articles_df = pd.DataFrame()
except ValueError as ve:
    print(f"ERROR: {ve}")
    articles_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred loading the CSV: {e}")
    articles_df = pd.DataFrame()


# --- 2. Load Model and Tokenizer ---
articles_model = None
articles_tokenizer = None
if not articles_df.empty:
    try:
        print(f"\nLoading model and tokenizer from {MODEL_PATH_FOR_ARTICLES_CSV}...")
        articles_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH_FOR_ARTICLES_CSV)
        articles_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH_FOR_ARTICLES_CSV)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        articles_model.to(device)
        articles_model.eval()
        print("Model and tokenizer loaded successfully for article processing.")
    except Exception as e:
        print(f"Error loading model/tokenizer: {e}. Cannot proceed.")
else:
    print("Skipping model/tokenizer loading as no articles were loaded.")


# --- 3. Process Each Article ---
all_extracted_lulc_info = []
if not articles_df.empty and articles_model and articles_tokenizer and sent_tokenize:
    print(f"\n--- Processing {len(articles_df)} Articles ---")
    
    for index, row in articles_df.iterrows():
        current_article_id = row.get(ARTICLE_ID_COLUMN, f"Article_{index+1}")
        print(f"\nProcessing {current_article_id}...")

        # CORRECTED: Collect all text parts first
        combined_text_parts = []
        for col in TEXT_COLUMNS_TO_CONCATENATE:
            text_part = row.get(col, '')
            if pd.notna(text_part) and isinstance(text_part, str):
                combined_text_parts.append(text_part.strip()) # Strip individual parts too
        
        if not combined_text_parts:
            print(f"  No text found in specified columns for {current_article_id}. Skipping.")
            continue
            
        # CORRECTED: Join all parts together
        raw_combined_text = " ".join(combined_text_parts)
        
        # CORRECTED: Apply preprocessing to the full combined text
        # print(f"  Raw combined text length: {len(raw_combined_text)}") # Optional debug
        preprocessed_combined_text = preprocess_article_text(raw_combined_text)
        # print(f"  Preprocessed combined text length: {len(preprocessed_combined_text)}") # Optional debug

        if not preprocessed_combined_text.strip():
            print(f"  Text became empty after preprocessing for {current_article_id}. Skipping.")
            continue

        # Sentence Segmentation on preprocessed text
        try:
            article_sentences = sent_tokenize(preprocessed_combined_text)
        except Exception as e_seg:
            print(f"  Error segmenting sentences for {current_article_id}: {e_seg}. Skipping this article.")
            continue
        
        if not article_sentences:
            print(f"  No sentences found after segmentation for {current_article_id}. Skipping.")
            continue

        print(f"  Segmented into {len(article_sentences)} sentences. Classifying...")
        
        article_specific_lulc_sentences = 0 # Count for current article
        for sentence in article_sentences:
            clean_sentence = sentence.strip()
            if not clean_sentence:
                continue

            inputs = articles_tokenizer(
                clean_sentence,
                padding=True, truncation=True,
                max_length=PRED_MAX_LENGTH_FOR_ARTICLES_CSV,
                return_tensors="pt"
            )
            inputs = {k: v.to(articles_model.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = articles_model(**inputs)
                prediction = torch.argmax(outputs.logits, dim=-1).item()

            if prediction == 1: # LULC related
                all_extracted_lulc_info.append({
                    'article_id': current_article_id,
                    'lulc_sentence': clean_sentence
                })
                article_specific_lulc_sentences += 1
        
        print(f"  Found {article_specific_lulc_sentences} LULC-related sentences in {current_article_id}.")

    print("\n--- Finished Processing All Articles ---")
else:
    # More specific messages about why processing might be skipped
    if articles_df.empty: print("Skipping processing: No articles loaded from CSV.")
    elif not articles_model: print("Skipping processing: Model not loaded.")
    elif not articles_tokenizer: print("Skipping processing: Tokenizer not loaded.")
    elif not sent_tokenize: print("Skipping processing: Sentence tokenizer (NLTK) not available.")


# --- 4. Display or Save Results ---
if all_extracted_lulc_info:
    results_df = pd.DataFrame(all_extracted_lulc_info)
    print(f"\n--- All Extracted LULC-Related Sentences ({len(results_df)} total) ---")
    # To see more, you can print results_df without .head() or change the number
    if len(results_df) > 0 :
        print(results_df.head(20))
    else:
        print("DataFrame of results is empty (though all_extracted_lulc_info was not). This is unusual.")


    # Uncomment to save:
    results_df.to_csv("extracted_lulc_sentences_from_articles.csv", index=False)
    print("\nAll extracted LULC sentences saved to 'extracted_lulc_sentences_from_articles.csv'")
else:
    print("\nNo LULC-related sentences were identified in any of the processed articles.")

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1'}. The number of labels wil be overwritten to 2.


Loading articles from CSV: extracted_data_full.csv
Successfully loaded 118 articles.

Loading model and tokenizer from LULC_BART_final_model...
Model and tokenizer loaded successfully for article processing.

--- Processing 118 Articles ---

Processing Article_1...
  Segmented into 76 sentences. Classifying...
  Found 16 LULC-related sentences in Article_1.

Processing Article_2...
  Segmented into 88 sentences. Classifying...
  Found 43 LULC-related sentences in Article_2.

Processing Article_3...
  Segmented into 70 sentences. Classifying...
  Found 5 LULC-related sentences in Article_3.

Processing Article_4...
  Segmented into 64 sentences. Classifying...
  Found 17 LULC-related sentences in Article_4.

Processing Article_5...
  Segmented into 73 sentences. Classifying...
  Found 10 LULC-related sentences in Article_5.

Processing Article_6...
  Segmented into 62 sentences. Classifying...
  Found 9 LULC-related sentences in Article_6.

Processing Article_7...
  Segmented into 103 s

  Found 5 LULC-related sentences in Article_67.

Processing Article_68...
  Segmented into 59 sentences. Classifying...
  Found 18 LULC-related sentences in Article_68.

Processing Article_69...
  Segmented into 106 sentences. Classifying...
  Found 49 LULC-related sentences in Article_69.

Processing Article_70...
  Segmented into 27 sentences. Classifying...
  Found 10 LULC-related sentences in Article_70.

Processing Article_71...
  Segmented into 119 sentences. Classifying...
  Found 12 LULC-related sentences in Article_71.

Processing Article_72...
  Segmented into 82 sentences. Classifying...
  Found 11 LULC-related sentences in Article_72.

Processing Article_73...
  Segmented into 6 sentences. Classifying...
  Found 1 LULC-related sentences in Article_73.

Processing Article_74...
  Segmented into 97 sentences. Classifying...
  Found 2 LULC-related sentences in Article_74.

Processing Article_75...
  Segmented into 65 sentences. Classifying...
  Found 10 LULC-related sentences 

In [12]:
# Cell 9: Process Multiple Articles from CSV to Extract and Save All Segmented Sentences (WITH DEBUGGING)

import pandas as pd
import re
from nltk.tokenize import sent_tokenize

CSV_FILE_PATH = 'extracted_data_full.csv'
# !!! REPLACE these with the actual names of your text columns !!!
TEXT_COLUMNS_TO_CONCATENATE = ['title', 'abstract', 'sections'] # Example names

ARTICLE_ID_COLUMN = None   # Example name

# --- Preprocessing Function Definition ---
def preprocess_article_text(text):
    if not isinstance(text, str): # Handle non-string inputs gracefully
        return ""
        
    # 1. Basic whitespace normalization
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Remove [1], [23] style citations
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text) # Handles [1], [1,2], [1, 2, 3]
    
    # 3. Attempt to remove (Author, YYYY) or (Author et al., YYYY)
    # This regex is simple and might need refinement for complex cases
    text = re.sub(r"\(\s*[\w\s\-\.']+(?:et\s+al\.?)?,\s*\d{4}\s*\)", '', text) 
    
    # 4. Example: Remove URLs (add this if URLs are common in your text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 5. ADD OTHER PREPROCESSING STEPS HERE AS NEEDED
    # (e.g., specific pattern removal for '#text': ..., HTML cleaning if source is web)
    
    return text

# --- 1. Load Articles from CSV ---
try:
    print(f"Loading articles from CSV: {CSV_FILE_PATH}")
    articles_df = pd.read_csv(CSV_FILE_PATH)
    print(f"Successfully loaded {len(articles_df)} articles.")
    
    # DEBUG: Check what columns actually exist
    print(f"DEBUG: Available columns in CSV: {list(articles_df.columns)}")
    print(f"DEBUG: First few rows of data:")
    print(articles_df.head(2))
    
    missing_text_cols = [col for col in TEXT_COLUMNS_TO_CONCATENATE if col not in articles_df.columns]
    if missing_text_cols:
        raise ValueError(f"Missing text columns in CSV: {missing_text_cols}. Expected: {TEXT_COLUMNS_TO_CONCATENATE}")
    if ARTICLE_ID_COLUMN and ARTICLE_ID_COLUMN not in articles_df.columns:
        print(f"Warning: Article ID column '{ARTICLE_ID_COLUMN}' not found. Using row index for article identification.")
except FileNotFoundError:
    print(f"ERROR: CSV file not found at {CSV_FILE_PATH}.")
    articles_df = pd.DataFrame()
except ValueError as ve:
    print(f"ERROR: {ve}")
    articles_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred loading the CSV: {e}")
    articles_df = pd.DataFrame()

# DEBUG: Check if sent_tokenize is available
try:
    test_sentence = "This is a test. This is another sentence."
    test_result = sent_tokenize(test_sentence)
    print(f"DEBUG: sent_tokenize is working. Test result: {test_result}")
except Exception as e:
    print(f"DEBUG: sent_tokenize error: {e}")
    print("You may need to run: import nltk; nltk.download('punkt')")

# --- 2. Process Each Article (No Model Loading Required) ---
all_segmented_sentences = []
if not articles_df.empty and 'sent_tokenize' in globals():
    print(f"\n--- Processing {len(articles_df)} Articles ---")
    
    # DEBUG: Process only first few articles to see what's happening
    for index, row in articles_df.iterrows():
        if index >= 200:  # Only process first 3 articles for debugging
            print(f"DEBUG: Stopping after 3 articles for debugging...")
            break
            
        current_article_id = row.get(ARTICLE_ID_COLUMN, f"Article_{index+1}")
        print(f"\nProcessing {current_article_id}...")

        # DEBUG: Check what's in each column
        print(f"DEBUG: Row data for {current_article_id}:")
        for col in TEXT_COLUMNS_TO_CONCATENATE:
            value = row.get(col, 'MISSING')
            if pd.isna(value):
                print(f"  {col}: NaN")
            else:
                print(f"  {col}: {type(value)} - {str(value)[:100]}...")

        # Collect all text parts first
        combined_text_parts = []
        for col in TEXT_COLUMNS_TO_CONCATENATE:
            text_part = row.get(col, '')
            if pd.notna(text_part) and isinstance(text_part, str):
                combined_text_parts.append(text_part.strip()) # Strip individual parts too
        
        print(f"DEBUG: Found {len(combined_text_parts)} non-empty text parts")
        
        if not combined_text_parts:
            print(f"  No text found in specified columns for {current_article_id}. Skipping.")
            continue
            
        # Join all parts together
        raw_combined_text = " ".join(combined_text_parts)
        print(f"DEBUG: Raw combined text length: {len(raw_combined_text)}")
        
        # Apply preprocessing to the full combined text
        preprocessed_combined_text = preprocess_article_text(raw_combined_text)
        print(f"DEBUG: Preprocessed text length: {len(preprocessed_combined_text)}")

        if not preprocessed_combined_text.strip():
            print(f"  Text became empty after preprocessing for {current_article_id}. Skipping.")
            continue

        # Sentence Segmentation on preprocessed text
        try:
            article_sentences = sent_tokenize(preprocessed_combined_text)
            print(f"DEBUG: Segmented into {len(article_sentences)} sentences")
        except Exception as e_seg:
            print(f"  Error segmenting sentences for {current_article_id}: {e_seg}. Skipping this article.")
            continue
        
        if not article_sentences:
            print(f"  No sentences found after segmentation for {current_article_id}. Skipping.")
            continue

        print(f"  Segmented into {len(article_sentences)} sentences.")
        
        # Save all segmented sentences
        sentences_added = 0
        for sentence_index, sentence in enumerate(article_sentences, 1):
            clean_sentence = sentence.strip()
            if not clean_sentence:
                continue

            all_segmented_sentences.append({
                'article_id': current_article_id,
                'sentence_number': sentence_index,
                'sentence': clean_sentence
            })
            sentences_added += 1
        
        print(f"  Saved {sentences_added} sentences from {current_article_id}.")

    print("\n--- Finished Processing Articles (DEBUG MODE) ---")
else:
    # More specific messages about why processing might be skipped
    if articles_df.empty: 
        print("Skipping processing: No articles loaded from CSV.")
    elif 'sent_tokenize' not in globals(): 
        print("Skipping processing: Sentence tokenizer (NLTK) not available.")

print(f"DEBUG: Total sentences collected: {len(all_segmented_sentences)}")

# --- 3. Display and Save Results ---
if all_segmented_sentences:
    results_df = pd.DataFrame(all_segmented_sentences)
    print(f"\n--- All Segmented Sentences ({len(results_df)} total) ---")
    # Display first 5 sentences as preview
    if len(results_df) > 0:
        print(results_df.head(5))
    else:
        print("DataFrame of results is empty (though all_segmented_sentences was not). This is unusual.")

    # Save all segmented sentences to CSV
    results_df.to_csv("all_segmented_sentences_from_articles.csv", index=False)
    print(f"\nAll {len(results_df)} segmented sentences saved to 'all_segmented_sentences_from_articles.csv'")
    
else:
    print("\nNo sentences were extracted from any of the processed articles.")
    print("DEBUG: Check the debug output above to see what went wrong.")

Loading articles from CSV: extracted_data_full.csv
Successfully loaded 118 articles.
DEBUG: Available columns in CSV: ['title', 'authors', 'abstract', 'sections', 'references']
DEBUG: First few rows of data:
                                               title  \
0  Land use and land cover change detection and p...   
1             Environmental & Socio-economic Studies   

                                        authors  \
0  Sonam Wang; Lamchin Munkhnasan; Woo-Kyun Lee   
1                             Tran Nguyen; Tuan   

                                            abstract  \
0  Rapid urbanization is changing landscapes ofte...   
1  Land-use change is a human process aimed at tr...   

                                            sections  \
0  Introduction: Global cities, which are the eng...   
1  Introduction: According to the literature, lan...   

                                          references  
0  Urban and Peri-urban agriculture in developing...  
1  Research and Appli

DEBUG: Segmented into 3189 sentences
  Segmented into 3189 sentences.
  Saved 3189 sentences from Article_33.

Processing Article_34...
DEBUG: Row data for Article_34:
  title: <class 'str'> - Analyzing and Predicting Land Use and Land Cover Changes in New Jersey Using Multi-Layer Perceptron-...
  abstract: <class 'str'> - This study analyzed the changes of land use and land cover (LULC) in New Jersey in the United States...
  sections: <class 'str'> - Introduction: Land use and land cover are two closely related concepts. Land cover consists of natur...
DEBUG: Found 3 non-empty text parts
DEBUG: Raw combined text length: 15637
DEBUG: Preprocessed text length: 15550
DEBUG: Segmented into 81 sentences
  Segmented into 81 sentences.
  Saved 81 sentences from Article_34.

Processing Article_35...
DEBUG: Row data for Article_35:
  title: <class 'str'> - Addressing uncertainty and bias in land use, land use change, and forestry greenhouse gas inventorie...
  abstract: <class 'str'> - Nation

DEBUG: Preprocessed text length: 9955
DEBUG: Segmented into 49 sentences
  Segmented into 49 sentences.
  Saved 49 sentences from Article_100.

Processing Article_101...
DEBUG: Row data for Article_101:
  title: <class 'str'> - Comparison of Land Use Land Cover Classifiers Using Different Satellite Imagery and Machine Learning...
  abstract: <class 'str'> - Accurate land use land cover (LULC) classification is vital for the sustainable management of natura...
  sections: <class 'str'> - Introduction: Accurate information on land use land cover (LULC) can facilitate various research act...
DEBUG: Found 3 non-empty text parts
DEBUG: Raw combined text length: 10258
DEBUG: Preprocessed text length: 10227
DEBUG: Segmented into 54 sentences
  Segmented into 54 sentences.
  Saved 54 sentences from Article_101.

Processing Article_102...
DEBUG: Row data for Article_102:
  title: <class 'str'> - ISPRS Journal of Photogrammetry and Remote Sensing...
  abstract: <class 'str'> - Products reveal th

In [13]:
# Cell: Configuration (Run this before helpers and Cell 10)
import logging
import os
import pandas as pd # For load_vocabulary_from_csv and load_data
import re # For preprocess_article_text if you use it, and pattern generation
import json # For convert_doc_to_doccano_json_file (though not used in Cell 10)

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Global Configuration Variables (Adjust as needed) ---
# Paths to your vocabulary CSV files
LULC_VOCAB_PATH = 'LULC.csv' # !!! REPLACE !!!
PROCESS_VOCAB_PATH = 'LCprocess.csv' # !!! REPLACE !!!
# VOCAB_TERM_COLUMN is used by load_vocabulary_from_csv
VOCAB_TERM_COLUMN = 'term'  # Or whatever your column name is for the terms

# Target base labels for your EntityRuler (ensure these are what you want to extract)
# This list is used by generate_ruler_patterns and add_ruler_to_nlp_pipeline
TARGET_BASE_LABELS = [
    "LULC", "PROCESS", "CHANGE", "SURFACE_UNIT", "COORDINATES",
    "GPE", "LOC", "DATE", "PERCENT", "QUANTITY", "CARDINAL", "RESEARCH_TERM" # Added RESEARCH_TERM as it's in your helpers
]
# Add any other custom labels your EntityRuler patterns might generate.

# Base SpaCy model
BASE_SPACY_MODEL = "en_core_web_sm" # Small model for speed, or use "en_core_web_md" / "en_core_web_lg" for better default NER

logging.info("Configuration cell executed.")

2025-06-30 15:01:32,669 - INFO - Configuration cell executed.


In [18]:
# Cell: Helper Functions (Full Version with Refined Coordinates)
# Run this cell after your Configuration Cell and before Cell 10.

import logging
import os
import pandas as pd
import re # For pattern generation and preprocess_article_text (if used later)
import json # For Doccano export (not directly used in Cell 10's NER display)
import spacy
from spacy.tokens import Doc # For Doc.has_extension, Doc.set_extension

# --- Add Doc ID extension if it doesn't exist (safe to run multiple times) ---
# This should ideally be run once when spacy is first imported.
# Placing it here ensures it's set if this cell is the first to import spacy.Doc heavily.
if not Doc.has_extension('doc_id'):
    Doc.set_extension('doc_id', default=None)
    logging.info("SpaCy Doc extension 'doc_id' registered.")

def load_vocabulary_from_csv(csv_path, term_column="term"): # Using VOCAB_TERM_COLUMN from global config
    """Loads terms from a specified column in a CSV file."""
    global VOCAB_TERM_COLUMN # Access global config if term_column not overridden
    # Use argument `term_column` if provided, else use global `VOCAB_TERM_COLUMN` if defined, else default
    effective_term_column = term_column
    if term_column == "term" and 'VOCAB_TERM_COLUMN' in globals(): # Prioritize arg, then global, then default
        effective_term_column = VOCAB_TERM_COLUMN

    logging.info(f"Attempting to load vocabulary from: {csv_path} using term column: '{effective_term_column}'")
    if not os.path.exists(csv_path):
        logging.error(f"Vocabulary file not found at path: {csv_path}")
        return set()
    try:
        try:
            df_vocab = pd.read_csv(csv_path, encoding='utf-8')
        except UnicodeDecodeError:
            logging.warning(f"UTF-8 decoding failed for {csv_path}, trying 'latin1' encoding.")
            df_vocab = pd.read_csv(csv_path, encoding='latin1')

        if effective_term_column not in df_vocab.columns:
            logging.error(f"Error: Column '{effective_term_column}' not found in vocabulary file: {csv_path}")
            logging.error(f"Available columns: {df_vocab.columns.tolist()}")
            return set()

        vocab_set = set(df_vocab[effective_term_column].dropna().astype(str).str.lower().str.strip().unique())
        vocab_set.discard('')
        logging.info(f"Successfully loaded {len(vocab_set)} unique terms from {csv_path} (column: '{effective_term_column}')")
        return vocab_set
    except Exception as e:
        logging.error(f"Error loading vocabulary from {csv_path}: {e}", exc_info=True)
        return set()

def load_data(csv_path, limit=None):
    """Loads data from CSV, combines text columns, and applies a limit."""
    # (Your existing load_data function - keeping it as you provided)
    logging.info(f"Attempting to load data from: {csv_path}")
    if not os.path.exists(csv_path):
        logging.error(f"Data file not found at path: {csv_path}")
        return None
    try:
        try:
            df = pd.read_csv(csv_path, encoding='utf-8')
        except UnicodeDecodeError:
            logging.warning(f"UTF-8 decoding failed for {csv_path}, trying 'latin1' encoding.")
            df = pd.read_csv(csv_path, encoding='latin1')

        logging.info(f"Loaded data from {csv_path}. Original shape: {df.shape}")
        text_columns_candidates = ['full_text', 'text', 'Text', 'content', 'Content', 'abstract', 'title', 'sections']
        available_columns = [col for col in text_columns_candidates if col in df.columns]

        if not available_columns:
             raise ValueError(f"Could not find suitable text columns among {text_columns_candidates}.")
        logging.info(f"Using text columns for 'full_text': {', '.join(available_columns)}")

        for col in available_columns:
             if col not in df.columns: continue # Should not happen due to check above
             df[col] = df[col].fillna('')

        # Ensure available_columns still point to existing columns after potential modifications (unlikely here)
        df['full_text'] = df[available_columns].astype(str).apply(' '.join, axis=1)
        df['full_text'] = df['full_text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
        logging.info("Combined text columns into 'full_text'.")

        if 'full_text' not in df.columns or df['full_text'].empty:
             raise ValueError("Failed to create or populate 'full_text' column or it's empty.")

        if 'doc_id' not in df.columns:
             logging.warning("No 'doc_id' column found, creating one from index.")
             df = df.reset_index().rename(columns={'index': 'doc_id'})
        else:
             df['doc_id'] = df['doc_id'].astype(str) # Ensure doc_id is string

        df = df[['doc_id', 'full_text']].copy() # Select only necessary columns
        logging.info(f"DataFrame columns after selection: {df.columns.tolist()}")

        if limit is not None and isinstance(limit, int) and limit > 0 and limit < len(df):
            logging.warning(f"Limiting data to first {limit} articles for debugging.")
            df = df.head(limit).copy()

        if df.empty or 'full_text' not in df.columns or 'doc_id' not in df.columns:
             raise ValueError("DataFrame is empty or missing essential columns ('full_text', 'doc_id') after processing and limiting.")
        logging.info(f"Processed DataFrame shape: {df.shape}")
        if not df.empty:
             logging.info(f"Example doc_id: {df['doc_id'].iloc[0]}, Example text start: '{df['full_text'].iloc[0][:200]}...'")
        return df
    except Exception as e:
        logging.error(f"Error loading/processing data CSV from {csv_path}: {e}", exc_info=True)
        return None

def _generate_patterns_for_single_vocab(nlp, vocab_set, label):
    """Generates patterns (LOWER and LEMMA) for terms in a vocabulary set."""
    # (Your existing _generate_patterns_for_single_vocab - keeping as you provided)
    vocab_patterns = []
    has_lemmatizer = nlp.has_pipe("lemmatizer")
    if not has_lemmatizer:
         logging.warning(f"SpaCy model '{nlp.meta.get('name', 'Unknown Model')}' does not appear to have a lemmatizer pipe. Will skip generating LEMMA patterns for '{label}' vocabulary.")

    for term in sorted(list(vocab_set)):
        if not term or not isinstance(term, str):
             logging.warning(f"Skipping invalid term in '{label}' vocabulary: {term}")
             continue
        try:
            doc_term = nlp(term.lower())
            if len(doc_term) == 0:
                logging.warning(f"Skipped '{label}' term '{term}': SpaCy tokenization resulted in an empty Doc.")
                continue
            lower_pattern = [{"LOWER": token.lower_} for token in doc_term if token.text.strip()]
            if lower_pattern:
                vocab_patterns.append({"label": label, "pattern": lower_pattern})
            else:
                logging.warning(f"Skipped '{label}' term '{term}': Could not generate valid LOWER patterns after tokenization/stripping.")
            if has_lemmatizer:
                 lemma_pattern = [{"LEMMA": token.lemma_} for token in doc_term if token.text.strip() and token.lemma_ and token.lemma_ != "-"]
                 if lemma_pattern:
                     vocab_patterns.append({"label": label, "pattern": lemma_pattern})
                 else:
                     logging.warning(f"Skipped '{label}' term '{term}': Could not generate valid LEMMA patterns (check tokenization/lemmatization results).")
        except Exception as e:
             logging.warning(f"Error generating patterns for '{label}' term '{term}': {e}")
    logging.info(f"Generated {len(vocab_patterns)} total patterns (LOWER + LEMMA attempts) for '{label}' vocabulary.")
    return vocab_patterns


def generate_ruler_patterns(nlp_tokenizer_like, lulc_vocab, process_vocab, extra_labels=None):
    """
    Generates a list of SpaCy patterns from vocabulary (LOWER and LEMMA sequences)
    and specific rules.
    """
    patterns = []
    global TARGET_BASE_LABELS # To access the global list defined in Config cell

    logging.info("Generating patterns from vocabularies...")
    try:
        # Ensure label is in TARGET_BASE_LABELS before generating
        if "LULC" in TARGET_BASE_LABELS:
            patterns.extend(_generate_patterns_for_single_vocab(nlp_tokenizer_like, lulc_vocab, "LULC"))
        else:
            logging.warning("Skipping LULC vocab patterns: 'LULC' not in TARGET_BASE_LABELS.")
        if "PROCESS" in TARGET_BASE_LABELS:
            patterns.extend(_generate_patterns_for_single_vocab(nlp_tokenizer_like, process_vocab, "PROCESS"))
        else:
            logging.warning("Skipping PROCESS vocab patterns: 'PROCESS' not in TARGET_BASE_LABELS.")
    except Exception as e:
        logging.error(f"Error generating vocabulary patterns: {e}", exc_info=True)

    # --- Specific Rule-Based Patterns ---
    if "SURFACE_UNIT" in TARGET_BASE_LABELS:
        logging.info("Generating SURFACE_UNIT patterns...")
        surface_unit_rule_patterns = [ # Copied from your example
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}],
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["sq", "square"]}, "OP":"?"}, {"LOWER": {"IN": ["km", "kms", "kilometer", "kilometers", "m", "meter", "meters", "mi", "miles", "yd", "yards", "ft", "feet", "cm", "centimeter", "centimeters", "ft."]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["sq", "square"]}, "OP":"?"}, {"LOWER": {"IN": ["km", "kms", "kilometer", "kilometers", "m", "meter", "meters", "mi", "miles", "yd", "yards", "ft", "feet", "cm", "centimeter", "centimeters", "ft."]}}],
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["acres", "acre"]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["acres", "acre"]}}],
            [{"LIKE_NUM": True}, {"TEXT": {"REGEX": r"(km|m|mi|yd|ft|cm)[\u00B2\^2]"}}],
            [{"LOWER": {"IN": ["million", "billion", "thousand"]}, "OP": "?"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "sqkm", "km", "m", "mi", "yd", "ft", "cm", "acres"]}}],
            [{"LOWER": {"IN": ["area", "extent", "size", "covering"]}}, {"LOWER": "of", "OP": "?"}, {"OP":"{1,4}"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": "area"}, {"LOWER": "of", "OP":"?"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": {"IN": ["covering", "extent"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": {"IN": ["approx.", "approximately", "around", "about"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LIKE_NUM": True}, {"TEXT": {"REGEX": r"[\u2013\u2014-]|to|and"}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
            [{"LOWER": {"IN": ["between"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["and", "-"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectares", "acres", "acre", "sqkm", "km"]}}],
        ]
        for pattern in surface_unit_rule_patterns:
            if isinstance(pattern, list) and all(isinstance(p, dict) for p in pattern):
                 patterns.append({"label": "SURFACE_UNIT", "pattern": pattern})
            else: logging.warning(f"Skipping invalid SURFACE_UNIT pattern: {pattern}")
        logging.info(f"Generated {len([p for p in patterns if p.get('label') == 'SURFACE_UNIT'])} SURFACE_UNIT patterns.")
    else:
        logging.warning("Skipping SURFACE_UNIT patterns: Not in TARGET_BASE_LABELS.")


    if "COORDINATES" in TARGET_BASE_LABELS:
        logging.info("Generating COORDINATES patterns (refined)...")
        coordinate_rule_patterns = []
        num_token_regex = r"^[+-]?\d+(\.\d+)?$" # Matches numbers like 45, 45.123, -45.123

        # Pattern 1: Decimal Degrees with N/S/E/W suffix, comma separated
        coordinate_rule_patterns.append([
            {"TEXT": {"REGEX": num_token_regex}},
            {"LOWER": {"REGEX": r"^[ns]$"}, "OP": "?"},
            {"LOWER": {"REGEX": r"^(north|south|n|s\.?)$"}},
            {"TEXT": ",", "OP": "?"},
            {"TEXT": {"REGEX": num_token_regex}},
            {"LOWER": {"REGEX": r"^[ew]$"}, "OP": "?"},
            {"LOWER": {"REGEX": r"^(east|west|e|w\.?)$"}}
        ])
        # Pattern 2: Decimal Degrees with ° symbol and N/S/E/W, various separators
        degree_symbol = "°" 
        coordinate_rule_patterns.append([
            {"TEXT": {"REGEX": num_token_regex}}, {"TEXT": degree_symbol},
            {"LOWER": {"REGEX": r"^[ns]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(north|south|n|s\.?)$"}},
            {"TEXT": {"REGEX": r"^([\–\-yto]|to)$"}, "OP": "?"}, # Corrected regex for separators
            {"TEXT": {"REGEX": num_token_regex}}, {"TEXT": degree_symbol},
            {"LOWER": {"REGEX": r"^[ew]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(east|west|e|w\.?)$"}}
        ])
        # Pattern 3: Keyword prefixed coordinates
        lat_keywords = ["lat", "latitude", "lat."]; lon_keywords = ["lon", "long", "longitude", "long."]; coord_keywords = ["coordinates", "coords", "position", "location", "center", "centroid"]
        coordinate_rule_patterns.append([
            {"LOWER": {"IN": lat_keywords}}, {"TEXT": {"IN": [":", "is", "at"]}, "OP": "?"}, {"OP":"{0,2}"}, # allow few words like 'is at approx'
            {"TEXT": {"REGEX": num_token_regex}},
            {"TEXT": {"IN": [",", ";"]}, "OP": "?"}, 
            {"LOWER": {"IN": lon_keywords}, "OP": "?"}, {"TEXT": {"IN": [":", "is", "at"]}, "OP": "?"}, {"OP":"{0,2}"},
            {"TEXT": {"REGEX": num_token_regex}}
        ])
        coordinate_rule_patterns.append([
            {"LOWER": {"IN": coord_keywords}}, {"TEXT": {"IN": [":", "is", "at"]}, "OP": "?"}, {"OP":"{0,2}"},
            {"TEXT": {"REGEX": num_token_regex}},
            {"TEXT": {"IN": [",", ";", "/"]}, "OP": "?"},
            {"TEXT": {"REGEX": num_token_regex}}
        ])
        # Pattern 4: N/S/E/W prefixes for numbers (single token like N45.123)
        ns_prefix_num_regex = r"^[nsNS]([+-]?\d+(\.\d+)?)$"
        ew_prefix_num_regex = r"^[ewEW]([+-]?\d+(\.\d+)?)$"
        coordinate_rule_patterns.append([
            {"TEXT": {"REGEX": ns_prefix_num_regex}}, {"TEXT": { "IN": [",", ";"]}, "OP": "?"},
            {"TEXT": {"REGEX": ew_prefix_num_regex}}
        ])
        coordinate_rule_patterns.append([ # Order swapped
            {"TEXT": {"REGEX": ew_prefix_num_regex}}, {"TEXT": { "IN": [",", ";"]}, "OP": "?"},
            {"TEXT": {"REGEX": ns_prefix_num_regex}}
        ])
        # Pattern 5 (DMS - simplified to avoid excessive FPs, still risky)
        # Matches NUMBER symbol NUMBER symbol NUMBER symbol DIRECTION (e.g. 40 ° 20 ' 10 " N)
        # This requires the symbols to be separate tokens, which is common.
        dms_symbols_tokens = ["°", "d", "'", "m", "\"", "s" ] # common symbols as separate tokens
        coordinate_rule_patterns.append([
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"}, # Degree value and optional symbol
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"}, # Minute value and optional symbol
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"}, # Second value and optional symbol
            {"LOWER": {"REGEX": r"^[ns]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(north|south|n|s\.?)$"}}
        ])
        coordinate_rule_patterns.append([ # For Longitude DMS part
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"},
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"},
            {"LIKE_NUM": True}, {"LOWER": {"IN": dms_symbols_tokens}, "OP": "?"},
            {"LOWER": {"REGEX": r"^[ew]$"}, "OP": "?"}, {"LOWER": {"REGEX": r"^(east|west|e|w\.?)$"}}
        ])
        logging.warning("DMS patterns for coordinates are approximate and might need further refinement or a dedicated parser.")
        # Pattern 6: Two LIKE_NUM tokens (broad, last resort, only if other rules don't catch)
        # Example: if preceded by "located at" or "position" (can be made more specific)
        coordinate_rule_patterns.append([
            {"LOWER": {"IN": ["at", "is", "point"]}, "OP": "?"}, # context keyword
            {"LIKE_NUM": True},
            {"TEXT": {"IN": [",", ";", "/", "-"]}, "OP": "?"},
            {"LIKE_NUM": True}
        ])
        logging.warning("General paired LIKE_NUM patterns for coordinates are broad; context is key.")

        num_coord_patterns_added = 0
        for pattern_definition in coordinate_rule_patterns:
            if isinstance(pattern_definition, list) and len(pattern_definition) > 0 and \
               all(isinstance(token_matcher, dict) for token_matcher in pattern_definition):
                patterns.append({"label": "COORDINATES", "pattern": pattern_definition})
                num_coord_patterns_added +=1
            else:
                logging.warning(f"Skipping invalid COORDINATES pattern structure: {pattern_definition}")
        logging.info(f"Generated {num_coord_patterns_added} COORDINATES patterns from rules.")
    else:
        logging.warning("Skipping COORDINATES patterns: Not in TARGET_BASE_LABELS.")

    if "CHANGE" in TARGET_BASE_LABELS:
        logging.info("Generating CHANGE patterns...")
        change_lemma_terms = ["increase", "decrease", "loss", "gain", "expansion", "expand", "reduction", "grow", "growth", "change", "decline", "improve", "transform", "vary", "fluctuate", "shift", "transition", "convert", "alter", "mutation", "modify", "impact", "effect", "cause", "drive", "shift", "driver"]
        change_rule_patterns_list = [] # Use a temporary list for this section's patterns

        has_lemmatizer_for_change = nlp_tokenizer_like.has_pipe("lemmatizer")
        if has_lemmatizer_for_change:
             change_rule_patterns_list.append({"label": "CHANGE", "pattern": [{"LEMMA": {"IN": change_lemma_terms}}]})
             logging.info("Added CHANGE lemma pattern (lemmatizer detected).")
        else:
             logging.warning(f"Skipping CHANGE lemma pattern: SpaCy model lacks a lemmatizer pipe.")

        # Add specific multi-token phrase patterns (your original ones)
        change_rule_patterns_list.extend([
             {"label": "CHANGE", "pattern": [{"LOWER": {"IN": ["as", "of", "in", "to", "for"]}}, {"LOWER": "change"}]},
             # Add more multi-token patterns for CHANGE here if you have them
        ])
        for pattern_dict in change_rule_patterns_list:
            patterns.append(pattern_dict) # Append valid dict directly
        logging.info(f"Generated {len([p for p in patterns if p.get('label') == 'CHANGE'])} CHANGE patterns.")
    else:
        logging.warning("Skipping CHANGE patterns: Not in TARGET_BASE_LABELS.")

    if "RESEARCH_TERM" in TARGET_BASE_LABELS:
        candidate_label = "RESEARCH_TERM" # Your original label
        logging.info(f"Adding patterns for '{candidate_label}' (candidate)...")
        # Using the structure from your file for RESEARCH_TERM
        candidate_patterns_list = [
            {"label": candidate_label, "pattern": [{"LOWER": {"IN": ["candidate", "candidates"]}}]},
            {"label": candidate_label, "pattern": [{"LOWER": {"IN": ["candidate", "candidates"]}}, {"LOWER": {"IN": ["area", "areas", "site", "sites", "location", "locations"]}}]},
            {"label": candidate_label, "pattern": [{"LOWER": {"IN": ["candidate", "candidates"]}}, {"LOWER": "for"}, {"OP":"+"}]},
        ]
        for pattern_dict in candidate_patterns_list:
             patterns.append(pattern_dict)
        logging.info(f"Added {len([p for p in patterns if p.get('label') == candidate_label])} '{candidate_label}' related patterns.")
    else:
         logging.warning(f"Skipping patterns for 'RESEARCH_TERM': Not in TARGET_BASE_LABELS.")

    if extra_labels: # Assuming extra_labels is a dict: {'LABEL_NAME': [pattern_list_for_label_1, ...]}
        logging.info(f"Processing extra patterns from extra_labels...")
        for label, label_patterns_list in extra_labels.items():
             if label not in TARGET_BASE_LABELS: # Assuming TARGET_BASE_LABELS is globally defined
                  logging.warning(f"Skipping extra patterns for label '{label}': Not in TARGET_BASE_LABELS.")
                  continue
             logging.info(f"Adding {len(label_patterns_list)} patterns for label '{label}'.")
             for pattern_list_content in label_patterns_list: # label_patterns_list contains the actual spaCy patterns
                 if isinstance(pattern_list_content, list) and len(pattern_list_content) > 0 and all(isinstance(p_token, dict) for p_token in pattern_list_content):
                      patterns.append({"label": label, "pattern": pattern_list_content})
                 else:
                      logging.warning(f"Skipping invalid extra pattern for label '{label}': {pattern_list_content}")
    else:
        logging.info("No extra patterns provided.")
    logging.info(f"Generated a total of {len(patterns)} patterns for EntityRuler.")
    return patterns

# --- SpaCy Pipeline Setup Functions (from your file) ---
def add_ruler_to_nlp_pipeline(nlp, patterns, ruler_name="entity_ruler"):
    # (Your existing add_ruler_to_nlp_pipeline function - keeping as you provided)
    if ruler_name in nlp.pipe_names:
         logging.info(f"Removing existing '{ruler_name}' pipe for ruler-only report setup.")
         nlp.remove_pipe(ruler_name)

    # Ensure TARGET_BASE_LABELS is accessible
    global TARGET_BASE_LABELS
    if 'TARGET_BASE_LABELS' not in globals():
        logging.error("FATAL: TARGET_BASE_LABELS not defined globally. Cannot filter patterns for ruler.")
        # Handle this error, perhaps by setting TARGET_BASE_LABELS to a default list of all unique labels in patterns
        # For now, let's assume it's defined from the Config cell.
        # return # or raise an error

    report_patterns = [p for p in patterns if p.get('label') in TARGET_BASE_LABELS]
    logging.info(f"Adding {len(report_patterns)} patterns to the ruler-only report pipeline (filtered by TARGET_BASE_LABELS).")

    if not report_patterns:
         logging.warning("No relevant patterns available for ruler-only report pipeline after filtering.")
         return
    try:
        ruler = nlp.add_pipe("entity_ruler", name=ruler_name, config={"overwrite_ents": True}, last=True)
        logging.info(f"Added new '{ruler_name}' pipe using add_pipe at the end.")
        ruler.add_patterns(report_patterns)
        logging.info(f"Standard SpaCy EntityRuler '{ruler_name}' configured with {len(report_patterns)} patterns.")
    except Exception as e:
        logging.error(f"Failed to add patterns to standard SpaCy EntityRuler '{ruler_name}': {e}", exc_info=True);
        raise e

def setup_nlp_pipeline_for_doccano(base_model_name, patterns_to_add, target_labels_for_ruler_rules):
    """Sets up a SpaCy NLP pipeline with a base model, its default 'ner' (if present),
    and an EntityRuler configured with the provided patterns for custom entities."""
    # (Your existing setup_nlp_pipeline_for_doccano function - keeping as you provided)
    nlp_for_doccano = None
    try:
        logging.info(f"Loading base SpaCy model '{base_model_name}' for Doccano processing (including default NER).")
        nlp_for_doccano = spacy.load(base_model_name)
        logging.info(f"SpaCy model '{base_model_name}' loaded. Default pipes: {nlp_for_doccano.pipe_names}")

        ruler_custom_patterns_filtered = [p for p in patterns_to_add if p.get('label') in target_labels_for_ruler_rules]
        if not ruler_custom_patterns_filtered:
            logging.warning(f"No patterns relevant to your custom target labels {target_labels_for_ruler_rules} found for the EntityRuler.")
        else:
            logging.info(f"Found {len(ruler_custom_patterns_filtered)} patterns for your custom EntityRuler.")

        ruler_pipe_name = "custom_entity_ruler"
        if ruler_pipe_name in nlp_for_doccano.pipe_names:
            logging.info(f"Removing existing '{ruler_pipe_name}' before adding a new one.")
            nlp_for_doccano.remove_pipe(ruler_pipe_name)

        # Determine position for the ruler
        ruler_config = {"overwrite_ents": True}
        if "ner" in nlp_for_doccano.pipe_names:
            logging.info(f"Adding EntityRuler '{ruler_pipe_name}' before default 'ner' pipe.")
            ruler = nlp_for_doccano.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
        else:
            logging.info(f"Adding EntityRuler '{ruler_pipe_name}' (default 'ner' pipe not found).")
            ruler = nlp_for_doccano.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
        
        if ruler_custom_patterns_filtered:
            ruler.add_patterns(ruler_custom_patterns_filtered)
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' configured with {len(ruler_custom_patterns_filtered)} patterns.")
        else:
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' added with 0 relevant patterns.")
            
        logging.info(f"Final pipeline for Doccano processing: {nlp_for_doccano.pipe_names}")
        return nlp_for_doccano
    except Exception as e:
        logging.error(f"FATAL: Failed to set up NLP pipeline for Doccano: {e}", exc_info=True)
        if nlp_for_doccano: del nlp_for_doccano # Clean up if part of it was loaded
        return None

def convert_doc_to_doccano_json_file(doc, original_text, output_dir, doc_id=None, target_labels=None):
    """Converts a SpaCy Doc to Doccano-compatible JSON format and saves it as a separate .json file."""
    # (Your existing convert_doc_to_doccano_json_file function - keeping as you provided)
    if doc is None:
        logging.warning("Input SpaCy Doc is None. Cannot convert to Doccano format.")
        return

    def normalize_label(label): # Nested helper, ensure it's accessible or defined globally if used elsewhere
        return 'LOC' if label in ['GPE', 'NORP'] else label

    annotations = []
    for ent in doc.ents:
        label = normalize_label(ent.label_) # Use the nested or global normalize_label
        if target_labels and label not in target_labels: # Filter by target_labels if provided
            continue
        annotations.append([ent.start_char, ent.end_char, label])

    doccano_entry = {"text": original_text, "labels": annotations }
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    file_name = f"{doc_id}.json" if doc_id else f"doc_{hash(original_text)}.json"
    file_path = os.path.join(output_dir, file_name)
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(doccano_entry, f, ensure_ascii=False, indent=2)
        logging.info(f"Saved: {file_path}")
    except Exception as e:
        logging.error(f"Error saving file '{file_path}': {e}", exc_info=True)

logging.info("Helper functions (Data Loading, Ruler/Pattern Generation, SpaCy Pipeline Setup, Doccano Export) cell defined/re-defined.")

2025-06-30 15:03:50,177 - INFO - Helper functions (Data Loading, Ruler/Pattern Generation, SpaCy Pipeline Setup, Doccano Export) cell defined/re-defined.


In [19]:
# Cell 10: NER on Sentences Loaded from CSV (Saving as JSON)

import spacy
from spacy.tokens import Doc
import pandas as pd
import logging
import json # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< IMPORT JSON MODULE

# --- Configuration for this Cell ---
SENTENCES_CSV_PATH = "all_segmented_sentences_from_articles.csv"
ARTICLE_ID_COL_IN_SENT_CSV = 'article_id'
SENTENCE_TEXT_COL_IN_SENT_CSV = 'sentence'
# This will be the name of your output JSON file
OUTPUT_NER_JSON_PATH = "extracted_ALL_entities_structured.json" # <<<<<<<<<< DEFINE OUTPUT JSON PATH

# Ensure TARGET_BASE_LABELS, LULC_VOCAB_PATH, PROCESS_VOCAB_PATH, BASE_SPACY_MODEL are defined globally

# Helper function for label normalization
def normalize_label(label_str):
    if label_str in ['GPE', 'NORP']:
        return 'LOC'
    return label_str

# --- 0. Load LULC-Related Sentences from CSV ---
sentences_to_process_df = pd.DataFrame()
try:
    logging.info(f"Loading LULC-related sentences from: {SENTENCES_CSV_PATH}")
    sentences_to_process_df = pd.read_csv(SENTENCES_CSV_PATH)
    if ARTICLE_ID_COL_IN_SENT_CSV not in sentences_to_process_df.columns or \
       SENTENCE_TEXT_COL_IN_SENT_CSV not in sentences_to_process_df.columns:
        raise ValueError(f"CSV must contain columns '{ARTICLE_ID_COL_IN_SENT_CSV}' and '{SENTENCE_TEXT_COL_IN_SENT_CSV}'")
    logging.info(f"Successfully loaded {len(sentences_to_process_df)} sentences from {SENTENCES_CSV_PATH}.")
except FileNotFoundError:
    logging.error(f"ERROR: Sentences CSV file not found at {SENTENCES_CSV_PATH}.")
except ValueError as ve:
    logging.error(f"ERROR: {ve}")
except Exception as e:
    logging.error(f"An unexpected error occurred loading sentences CSV: {e}")

# --- 1. Load Vocabularies ---
lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH) if 'LULC_VOCAB_PATH' in globals() and LULC_VOCAB_PATH else set()
process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH) if 'PROCESS_VOCAB_PATH' in globals() and PROCESS_VOCAB_PATH else set()

# --- 2. Setup SpaCy NLP Pipeline ---
nlp_ner_pipeline = None
if 'generate_ruler_patterns' in globals() and 'BASE_SPACY_MODEL' in globals():
    try:
        logging.info(f"Loading base SpaCy model '{BASE_SPACY_MODEL}' for NER...")
        nlp_ner_pipeline = spacy.load(BASE_SPACY_MODEL)
        # ( ... rest of your pipeline setup from the previous full cell version ... )
        # This includes loading vocabs, generating patterns, adding entity ruler
        logging.info(f"SpaCy model '{BASE_SPACY_MODEL}' loaded. Default pipes: {nlp_ner_pipeline.pipe_names}")
        logging.info("Generating NER patterns...")
        all_patterns_for_ruler = generate_ruler_patterns(nlp_ner_pipeline, lulc_vocab, process_vocab)

        ruler_pipe_name = "custom_entity_ruler"
        if ruler_pipe_name in nlp_ner_pipeline.pipe_names:
            nlp_ner_pipeline.remove_pipe(ruler_pipe_name)
        
        ruler_config = {"overwrite_ents": True}
        if "ner" in nlp_ner_pipeline.pipe_names:
            ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
        else:
            ruler = nlp_ner_pipeline.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
        
        if all_patterns_for_ruler:
            ruler.add_patterns(all_patterns_for_ruler)
            logging.info(f"Custom EntityRuler '{ruler_pipe_name}' configured with {len(all_patterns_for_ruler)} patterns.")
        else:
            logging.warning(f"Custom EntityRuler '{ruler_pipe_name}' added, but no patterns generated.")
        logging.info(f"Final NER pipeline: {nlp_ner_pipeline.pipe_names}")

    except Exception as e:
        logging.error(f"FATAL: Failed to set up NLP pipeline for NER: {e}", exc_info=True)
        nlp_ner_pipeline = None
else:
    logging.error("Required globals 'generate_ruler_patterns' or 'BASE_SPACY_MODEL' not found.")
    nlp_ner_pipeline = None

# --- 3. Process Sentences and Extract Entities ---
extracted_entities_data_structured = [] # Renamed for clarity
if nlp_ner_pipeline and not sentences_to_process_df.empty:
    print(f"\n--- Extracting Entities from {len(sentences_to_process_df)} LULC-related Sentences (from CSV) ---")
    for index, row in sentences_to_process_df.iterrows():
        article_id = row.get(ARTICLE_ID_COL_IN_SENT_CSV, f"Row_{index}")
        sentence_text = row.get(SENTENCE_TEXT_COL_IN_SENT_CSV, "")

        if not isinstance(sentence_text, str) or not sentence_text.strip():
            continue

        doc = nlp_ner_pipeline(sentence_text)
        entities_in_sentence = []
        if doc.ents:
            for ent in doc.ents:
                normalized_ent_label = normalize_label(ent.label_)
                if 'TARGET_BASE_LABELS' in globals() and normalized_ent_label in TARGET_BASE_LABELS:
                    entities_in_sentence.append({
                        'text': ent.text,
                        'label': normalized_ent_label,
                        'start_char': ent.start_char,
                        'end_char': ent.end_char
                    })
        
        # Append entry even if no entities are found for that sentence, if you want to keep all sentences
        extracted_entities_data_structured.append({
             'article_id': article_id,
             'original_sentence': sentence_text,
             'entities': entities_in_sentence # This will be an empty list if no entities found
         })
    print("\n--- Finished Entity Extraction ---")
elif sentences_to_process_df.empty:
    print("Skipping entity extraction: No sentences loaded from the CSV file.")
elif not nlp_ner_pipeline:
    print("Skipping entity extraction: NER pipeline not set up.")

# --- 4. Display Sample Results and Save as JSON File ---
if extracted_entities_data_structured:
    print(f"\n--- Extracted Entities Data (showing first 3 entries) ---")
    for i, entry in enumerate(extracted_entities_data_structured[:3]):
        print(f"\nArticle ID: {entry['article_id']}")
        print(f"Sentence: {entry['original_sentence']}")
        print("Entities:")
        if entry['entities']:
            for entity in entry['entities']:
                print(f"  - Text: '{entity['text']}', Label: {entity['label']}, Start: {entity['start_char']}, End: {entity['end_char']}")
        else:
            print("  (No entities extracted for this sentence based on TARGET_BASE_LABELS)")
    
    # --- SAVE AS A SINGLE JSON FILE ---
    try:
        with open(OUTPUT_NER_JSON_PATH, 'w', encoding='utf-8') as f_json:
            json.dump(extracted_entities_data_structured, f_json, ensure_ascii=False, indent=4) # indent for readability
        print(f"\n--- Successfully saved structured NER data to: {OUTPUT_NER_JSON_PATH} (single JSON file) ---")
    except Exception as e_json_save:
        print(f"\nError saving structured NER data to JSON: {e_json_save}")

else:
    print("\nNo data from entity extraction to display or save.")

2025-06-30 15:03:59,179 - INFO - Loading LULC-related sentences from: all_segmented_sentences_from_articles.csv
2025-06-30 15:03:59,224 - INFO - Successfully loaded 13660 sentences from all_segmented_sentences_from_articles.csv.
2025-06-30 15:03:59,226 - INFO - Attempting to load vocabulary from: LULC.csv using term column: 'term'
2025-06-30 15:03:59,230 - INFO - Successfully loaded 172 unique terms from LULC.csv (column: 'term')
2025-06-30 15:03:59,231 - INFO - Attempting to load vocabulary from: LCprocess.csv using term column: 'term'
2025-06-30 15:03:59,235 - INFO - Successfully loaded 19 unique terms from LCprocess.csv (column: 'term')
2025-06-30 15:03:59,236 - INFO - Loading base SpaCy model 'en_core_web_sm' for NER...
2025-06-30 15:03:59,239 - ERROR - FATAL: Failed to set up NLP pipeline for NER: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.
Traceback (most recent call last):
  File "/tmp/ipykernel_2436368/59

Skipping entity extraction: NER pipeline not set up.

No data from entity extraction to display or save.


In [16]:
# Optimized FLAN-T5 Zero-Shot Prompt for LULC Event Extraction

import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import pandas as pd
import re 
import logging

# --- Configuration ---
NER_OUTPUT_JSON_PATH = "extracted_entities_structured.json"
FLAN_T5_MODEL_NAME = "google/flan-t5-large"  # Upgraded from base to large
FLAN_EVENTS_SIMPLIFIED_OUTPUT_PATH = "flan_t5_extracted_lulc_events_optimized.csv"
NUM_SENTENCES_TO_PROCESS = 20
DELIMITER_LLM = "|||" # The delimiter we asked FLAN-T5 to use
NO_EVENT_MARKER_LLM = "NO_EVENT" # Changed from NO_EVENT_FOUND for brevity

# --- 1. Load NER Output ---
# (Same loading logic as before)

# --- 2. Load FLAN-T5 Model and Tokenizer ---
# (Same model loading logic as before)

# --- 3. Define Optimized Zero-Shot Prompt Construction and Extraction Logic ---
def format_entities_for_prompt(entities_list_for_sentence): # Your existing helper
    if not entities_list_for_sentence: return "No specific entities pre-identified."
    return "\n".join([f"- \"{ent.get('text', 'N/A')}\" (Type: {ent.get('label', 'N/A')})" for ent in entities_list_for_sentence])

def construct_flan_t5_prompt_OPTIMIZED(current_sentence_text, current_entities_list):
    formatted_entities = format_entities_for_prompt(current_entities_list)
    
    # Extract LULC entities specifically for better context
    lulc_entities = [ent for ent in current_entities_list if ent.get('label') == 'LULC']
    lulc_text = ", ".join([f"\"{ent.get('text')}\"" for ent in lulc_entities]) if lulc_entities else "None identified"
    
    prompt = f"""Extract LULC change information from this sentence:
"{current_sentence_text}"

Identified entities:
{formatted_entities}

Instructions:
If the sentence describes a change in land use or land cover, extract:
1. FROM: The original land type
2. TO: The resulting land type
3. CHANGE: Words indicating change (e.g., increase, decrease, conversion)
4. PROCESS: The broader process (e.g., deforestation, urbanization)
5. MAGNITUDE: Any percentage or area measurement

Format your answer exactly like these examples:

Example 1: "Forest cover declined by 15% in the region."
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%

Example 2: "Agricultural land was converted to urban areas."
FROM: agricultural land
TO: urban areas
CHANGE: converted
PROCESS: urbanization
MAGNITUDE: 

Example 3: "The study examined biodiversity in tropical forests."
NO_EVENT

Your answer for the given sentence:
"""
    return prompt

def parse_structured_format(generated_text):
    """Parse the structured format with FROM:, TO:, etc. labels"""
    if NO_EVENT_MARKER_LLM in generated_text:
        return {
            "event_found": False,
            "from_lulc": "",
            "to_lulc": "",
            "change_indicator": "",
            "lulc_process": "",
            "magnitude": ""
        }
    
    # Extract each field using regex
    from_match = re.search(r'FROM:\s*(.*?)(?=\nTO:|$)', generated_text, re.DOTALL)
    to_match = re.search(r'TO:\s*(.*?)(?=\nCHANGE:|$)', generated_text, re.DOTALL)
    change_match = re.search(r'CHANGE:\s*(.*?)(?=\nPROCESS:|$)', generated_text, re.DOTALL)
    process_match = re.search(r'PROCESS:\s*(.*?)(?=\nMAGNITUDE:|$)', generated_text, re.DOTALL)
    magnitude_match = re.search(r'MAGNITUDE:\s*(.*?)(?=\n|$)', generated_text, re.DOTALL)
    
    # Extract values or default to empty string
    from_lulc = from_match.group(1).strip() if from_match else ""
    to_lulc = to_match.group(1).strip() if to_match else ""
    change_indicator = change_match.group(1).strip() if change_match else ""
    lulc_process = process_match.group(1).strip() if process_match else ""
    magnitude = magnitude_match.group(1).strip() if magnitude_match else ""
    
    # Determine if an event was found (at least one field has content)
    event_found = bool(from_lulc or to_lulc or change_indicator or lulc_process)
    
    return {
        "event_found": event_found,
        "from_lulc": from_lulc,
        "to_lulc": to_lulc,
        "change_indicator": change_indicator,
        "lulc_process": lulc_process,
        "magnitude": magnitude
    }

extracted_flan_events_data = []

if flan_model and flan_tokenizer and sentences_with_entities:
    logging.info(f"\n--- Performing LULC Event Extraction (Optimized Zero-Shot) on {len(sentences_with_entities)} sentences ---")
    
    for entry_idx, entry in enumerate(sentences_with_entities):
        if entry_idx > 0 and entry_idx % 5 == 0:
             logging.info(f"  Processed {entry_idx}/{len(sentences_with_entities)} entries...")

        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{entry_idx}")

        if not sentence_text.strip(): continue

        prompt_text = construct_flan_t5_prompt_OPTIMIZED(sentence_text, entities)
        
        event_data_row = {
            'article_id': article_id, 
            'original_sentence': sentence_text,
            'llm_raw_output': 'Not Generated Yet', 
            'event_found': False, 
            'from_lulc': "", 
            'to_lulc': "", 
            'change_indicator': "", 
            'lulc_process': "", 
            'magnitude_percent': "", 
            'magnitude_area': "", 
            'error': None
        }

        try:
            input_ids = flan_tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(flan_model.device)
            outputs = flan_model.generate(
                input_ids, 
                max_new_tokens=200,  # Increased for more complete responses
                num_beams=4,         # Increased for better quality
                temperature=0.3,     # Lower temperature for more deterministic outputs
                do_sample=False,     # Disable sampling for more consistent outputs
                early_stopping=True 
            )
            generated_text = flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
            event_data_row['llm_raw_output'] = generated_text
            
            # Parse the structured format response
            parsed_result = parse_structured_format(generated_text)
            
            event_data_row['event_found'] = parsed_result['event_found']
            event_data_row['from_lulc'] = parsed_result['from_lulc']
            event_data_row['to_lulc'] = parsed_result['to_lulc']
            event_data_row['change_indicator'] = parsed_result['change_indicator']
            event_data_row['lulc_process'] = parsed_result['lulc_process']
            
            # Process magnitude into percent and area
            raw_magnitude = parsed_result['magnitude']
            if "%" in raw_magnitude:
                event_data_row['magnitude_percent'] = raw_magnitude
            elif any(unit in raw_magnitude.lower() for unit in ["ha", "km", "acre", "meter", "sq", "hectare"]):
                event_data_row['magnitude_area'] = raw_magnitude
            elif re.search(r'\d+\s*(?:ha|km|m|acre)', raw_magnitude, re.IGNORECASE):
                event_data_row['magnitude_area'] = raw_magnitude
            elif raw_magnitude:
                # Try to determine if it's a percentage without % symbol
                if re.search(r'\d+\.\d+|\d+', raw_magnitude):
                    if float(re.search(r'\d+\.\d+|\d+', raw_magnitude).group()) <= 100:
                        event_data_row['magnitude_percent'] = raw_magnitude
                    else:
                        event_data_row['magnitude_area'] = raw_magnitude
                else:
                    event_data_row['magnitude_area'] = raw_magnitude
            
        except Exception as e_flan_gen:
            logging.error(f"Error during FLAN-T5 generation for '{article_id}': {e_flan_gen}", exc_info=True)
            event_data_row['error'] = f"Generation Error: {str(e_flan_gen)}"
            event_data_row['llm_raw_output'] = 'Error during generation'
        
        extracted_flan_events_data.append(event_data_row)

    logging.info(f"\n--- Finished FLAN-T5 Event Extraction (Optimized Zero-Shot). Processed {len(extracted_flan_events_data)} entries. ---")
else:
    # Logging for skipped processing
    pass


# --- 4. Display and Save Results ---
if extracted_flan_events_data:
    flan_results_df = pd.DataFrame(extracted_flan_events_data)
    logging.info(f"\n--- FLAN-T5 Extracted LULC Change Events (Optimized Zero-Shot) ---")
    if not flan_results_df.empty:
        display_cols_flan = ['article_id', 'original_sentence', 'event_found', 
                             'from_lulc', 'to_lulc', 'change_indicator', 'lulc_process',
                             'magnitude_percent', 'magnitude_area', 
                             'error', 'llm_raw_output'] # Keep llm_raw_output for inspection
        
        final_display_cols_flan = [col for col in display_cols_flan if col in flan_results_df.columns]
        
        # Calculate and display statistics
        events_found = flan_results_df['event_found'].sum()
        total_processed = len(flan_results_df)
        event_rate = (events_found / total_processed) * 100 if total_processed > 0 else 0
        
        logging.info(f"Events found: {events_found}/{total_processed} ({event_rate:.1f}%)")
        
        # Display field completion rates for found events
        if events_found > 0:
            events_subset = flan_results_df[flan_results_df['event_found']]
            for field in ['from_lulc', 'to_lulc', 'change_indicator', 'lulc_process']:
                field_completion = (events_subset[field].astype(bool).sum() / events_found) * 100
                logging.info(f"  {field} completion rate: {field_completion:.1f}%")
        
        with pd.option_context('display.max_colwidth', 80, 'display.max_rows', 20, 'display.width', 1000):
            print(flan_results_df[final_display_cols_flan].head(NUM_SENTENCES_TO_PROCESS if NUM_SENTENCES_TO_PROCESS else 10))

        try:
            flan_results_df[final_display_cols_flan].to_csv(FLAN_EVENTS_SIMPLIFIED_OUTPUT_PATH, index=False, encoding='utf-8')
            logging.info(f"Saved FLAN-T5 extracted events to: {FLAN_EVENTS_SIMPLIFIED_OUTPUT_PATH}")
        except Exception as e_save_flan:
            logging.error(f"Error saving FLAN-T5 results to CSV: {e_save_flan}")
    else:
        logging.info("FLAN-T5 results DataFrame is empty after processing.")
else:
    logging.info("\nNo LULC change events processed/extracted or process was skipped.")


pip install llama-stack

SyntaxError: invalid syntax (3600209924.py, line 235)

In [1]:
pip install llama-stack -U


Requirement already up-to-date: llama-stack in /home/raham/venv/lib/python3.8/site-packages (0.0.1a5)
Note: you may need to restart the kernel to use updated packages.


In [2]:
!llama_model list


/bin/bash: llama_model : commande introuvable


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
import torch

# Use Hugging Face gated model (requires accepted terms and authentication)
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model (BF16 for LLaMA 3 recommended)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Define conversation (chat template-compatible format)
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"}
]

# Tokenize input with chat template
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Custom stopping criteria for multiple EOS tokens
class MultiTokenEosCriteria(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = stop_ids

    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1].item() in self.stop_ids

# Define EOS token options
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# Generate output
outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    temperature=0.6,
    top_p=0.9,
    do_sample=True,
    stopping_criteria=StoppingCriteriaList([MultiTokenEosCriteria(terminators)])
)

# Extract and decode only the new tokens
response = outputs[0][input_ids.shape[-1]:]
print("\n🗨️ LLaMA 3 says:\n" + tokenizer.decode(response, skip_special_tokens=True))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



🗨️ LLaMA 3 says:
Arrrr, me hearty! Me name be Captain Chatbot, the scurviest chatbot to ever sail the Seven Seas! Me be a swashbucklin' AI, here to chat with ye about all manner o' things, from treasure maps to sea shanties. So hoist the colors, me matey, and let's set sail fer a chat!


In [4]:
def run_llama3_pipeline(
    input_path="extracted_entities_structured.json",
    output_path="llama3_output.csv",
    model_id="meta-llama/Meta-Llama-3-8B-Instruct",
    num_samples=0,
    quantize=False,
    device="cuda" if torch.cuda.is_available() else "cpu"
):
    class Args:
        def __init__(self, input, output, model, num_samples, quantize, device):
            self.input = input
            self.output = output
            self.model = model
            self.num_samples = num_samples
            self.quantize = quantize
            self.device = device

    args = Args(input_path, output_path, model_id, num_samples, quantize, device)

    # ⬇️ Paste your original `main()` function logic here, replacing parse_args() with `args`



In [7]:
"""
LLaMA 3 Implementation for LULC Event Extraction
================================================

This script demonstrates how to use LLaMA 3 for extracting Land Use Land Cover (LULC) 
change events from text. It includes:

1. Setup and installation instructions
2. Model loading and configuration
3. Optimized prompt design for LLaMA 3
4. Processing pipeline for LULC event extraction
5. Output formatting and evaluation

Requirements:
- Python 3.8+
- PyTorch 2.0+
- transformers 4.30.0+
- GPU with at least 16GB VRAM (for 8B model)
"""

import json
import os
import re
import logging
import argparse
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("llama3_lulc_extraction.log")
    ]
)

# Default paths and configuration
DEFAULT_NER_OUTPUT_PATH = "extracted_entities_structured.json"
DEFAULT_OUTPUT_PATH = "llama3_extracted_lulc_events.csv"
DEFAULT_MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # 8B version
# Alternative models:
# "meta-llama/Meta-Llama-3-8B-Instruct" - Instruction-tuned version
# "meta-llama/Meta-Llama-3-70B" - Larger model (requires more VRAM)

def parse_arguments():
    """Parse command line arguments."""
    parser = argparse.ArgumentParser(description="LLaMA 3 LULC Event Extraction")
    parser.add_argument("--input", type=str, default=DEFAULT_NER_OUTPUT_PATH,
                        help=f"Path to NER output JSON file (default: {DEFAULT_NER_OUTPUT_PATH})")
    parser.add_argument("--output", type=str, default=DEFAULT_OUTPUT_PATH,
                        help=f"Path to output CSV file (default: {DEFAULT_OUTPUT_PATH})")
    parser.add_argument("--model", type=str, default=DEFAULT_MODEL_ID,
                        help=f"LLaMA 3 model ID (default: {DEFAULT_MODEL_ID})")
    parser.add_argument("--num_samples", type=int, default=0,
                        help="Number of samples to process (0 for all)")
    parser.add_argument("--quantize", action="store_true",
                        help="Use 4-bit quantization to reduce memory usage")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu",
                        help="Device to use (default: cuda if available, else cpu)")
    return parser.parse_args()

def setup_model(model_id, device, use_quantization=False):
    """
    Load the LLaMA 3 model and tokenizer.
    
    Args:
        model_id: Hugging Face model ID
        device: Device to load the model on
        use_quantization: Whether to use 4-bit quantization
        
    Returns:
        model, tokenizer
    """
    logging.info(f"Loading LLaMA 3 tokenizer: {model_id}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Configure quantization if requested
    if use_quantization:
        logging.info("Using 4-bit quantization")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    else:
        quantization_config = None
    
    # Load model with appropriate configuration
    logging.info(f"Loading LLaMA 3 model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device,
        quantization_config=quantization_config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    
    logging.info(f"LLaMA 3 model loaded successfully on {device}")
    return model, tokenizer

def format_entities_for_prompt(entities):
    """Format entities for inclusion in the prompt."""
    if not entities:
        return "No specific entities pre-identified."
    
    # Group entities by type
    entities_by_type = {}
    for entity in entities:
        entity_type = entity.get('label', 'UNKNOWN')
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity.get('text', 'N/A'))
    
    # Format grouped entities
    formatted_lines = []
    for entity_type, entity_texts in entities_by_type.items():
        unique_texts = list(set(entity_texts))  # Remove duplicates
        # Fixed line to avoid nested f-string syntax error
        entity_list = ", ".join(['"' + text + '"' for text in unique_texts])
        formatted_lines.append(f"- {entity_type}: {entity_list}")
    
    return "\n".join(formatted_lines)

def construct_llama3_prompt(sentence_text, entities):
    """
    Construct an optimized prompt for LLaMA 3.
    
    This prompt is specifically designed for LLaMA 3's capabilities and includes:
    1. Clear task definition
    2. Structured format instructions
    3. Few-shot examples
    4. Entity context
    """
    formatted_entities = format_entities_for_prompt(entities)
    
    # Extract LULC entities specifically for better context
    lulc_entities = [ent for ent in entities if ent.get('label') == 'LULC']
    # Fixed line to avoid nested f-string syntax error
    lulc_text = ", ".join(['"' + ent.get("text") + '"' for ent in lulc_entities]) if lulc_entities else "None identified"
    
    # Extract change indicators for better context
    change_entities = [ent for ent in entities if ent.get('label') == 'CHANGE']
    # Fixed line to avoid nested f-string syntax error
    change_text = ", ".join(['"' + ent.get("text") + '"' for ent in change_entities]) if change_entities else "None identified"
    
    prompt = f"""<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
</context>

<input>
"{sentence_text}"
</input>

<entities>
{formatted_entities}
</entities>

<instructions>
Analyze the input text and extract a LULC change event with these components:

FROM: The original land use/cover type that is changing or being converted
TO: The resulting land use/cover type
CHANGE: Words indicating change (e.g., increase, decrease, conversion)
PROCESS: The broader process (e.g., deforestation, urbanization)
MAGNITUDE: Any percentage or area measurement

If no LULC change event is present, respond with "NO_EVENT".
</instructions>

<examples>
Example 1:
Input: "Forest cover declined by 15% in the region."
Entities:
- LULC: "Forest"
- CHANGE: "declined"
- PERCENT: "15%"
- LOC: "region"

Output:
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%

Example 2:
Input: "Agricultural land was converted to urban areas between 2010 and 2020."
Entities:
- LULC: "Agricultural land", "urban areas"
- CHANGE: "converted"
- DATE: "2010", "2020"

Output:
FROM: agricultural land
TO: urban areas
CHANGE: converted
PROCESS: urbanization
MAGNITUDE: 

Example 3:
Input: "The study examined biodiversity in tropical forests."
Entities:
- LULC: "tropical forests"

Output:
NO_EVENT

Example 4:
Input: "Built-up area increased from 52.88% in 2002 to 65.5% in 2018, a change of 12.77%."
Entities:
- LULC: "Built-up area"
- CHANGE: "increased"
- PERCENT: "52.88%", "65.5%", "12.77%"
- DATE: "2002", "2018"

Output:
FROM: built-up area
TO: built-up area
CHANGE: increased
PROCESS: urbanization
MAGNITUDE: 12.77%
</examples>

<output>
"""
    return prompt

def generate_with_llama3(model, tokenizer, prompt, max_new_tokens=256):
    """
    Generate text using LLaMA 3 model.
    
    Args:
        model: LLaMA 3 model
        tokenizer: LLaMA 3 tokenizer
        prompt: Input prompt
        max_new_tokens: Maximum number of tokens to generate
        
    Returns:
        Generated text
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate with appropriate parameters for structured extraction
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,  # Low temperature for deterministic outputs
            top_p=0.9,
            do_sample=True,  # Light sampling for better quality
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_output[len(tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)):]
    
    # Clean up the output
    generated_text = generated_text.strip()
    
    # If the output contains </output> tag, extract only the content before it
    if "</output>" in generated_text:
        generated_text = generated_text.split("</output>")[0].strip()
    
    return generated_text

def parse_llama3_output(output_text):
    """
    Parse the LLaMA 3 output into structured fields.
    
    Args:
        output_text: Raw output from LLaMA 3
        
    Returns:
        Dictionary with parsed fields
    """
    # Check for NO_EVENT marker
    if "NO_EVENT" in output_text:
        return {
            "event_found": False,
            "from_lulc": "",
            "to_lulc": "",
            "change_indicator": "",
            "lulc_process": "",
            "magnitude": ""
        }
    
    # Extract fields using regex
    from_match = re.search(r'FROM:\s*(.*?)(?=\nTO:|$)', output_text, re.DOTALL)
    to_match = re.search(r'TO:\s*(.*?)(?=\nCHANGE:|$)', output_text, re.DOTALL)
    change_match = re.search(r'CHANGE:\s*(.*?)(?=\nPROCESS:|$)', output_text, re.DOTALL)
    process_match = re.search(r'PROCESS:\s*(.*?)(?=\nMAGNITUDE:|$)', output_text, re.DOTALL)
    magnitude_match = re.search(r'MAGNITUDE:\s*(.*?)(?=\n|$)', output_text, re.DOTALL)
    
    # Extract values or default to empty string
    from_lulc = from_match.group(1).strip() if from_match else ""
    to_lulc = to_match.group(1).strip() if to_match else ""
    change_indicator = change_match.group(1).strip() if change_match else ""
    lulc_process = process_match.group(1).strip() if process_match else ""
    magnitude = magnitude_match.group(1).strip() if magnitude_match else ""
    
    # Determine if an event was found (at least one field has content)
    event_found = bool(from_lulc or to_lulc or change_indicator or lulc_process)
    
    return {
        "event_found": event_found,
        "from_lulc": from_lulc,
        "to_lulc": to_lulc,
        "change_indicator": change_indicator,
        "lulc_process": lulc_process,
        "magnitude": magnitude
    }

def process_magnitude(magnitude):
    """
    Process magnitude into percent and area components.
    
    Args:
        magnitude: Raw magnitude string
        
    Returns:
        Tuple of (magnitude_percent, magnitude_area)
    """
    if not magnitude:
        return "", ""
    
    magnitude_percent = ""
    magnitude_area = ""
    
    # Check for percentage
    if "%" in magnitude:
        magnitude_percent = magnitude
    # Check for area units
    elif any(unit in magnitude.lower() for unit in ["ha", "km", "acre", "meter", "sq", "hectare"]):
        magnitude_area = magnitude
    # Check for numbers with area units using regex
    elif re.search(r'\d+\s*(?:ha|km|m|acre)', magnitude, re.IGNORECASE):
        magnitude_area = magnitude
    # If it's just a number, try to determine if it's a percentage
    elif re.search(r'\d+\.\d+|\d+', magnitude):
        try:
            value = float(re.search(r'\d+\.\d+|\d+', magnitude).group())
            if value <= 100:
                magnitude_percent = magnitude
            else:
                magnitude_area = magnitude
        except:
            magnitude_area = magnitude
    else:
        magnitude_area = magnitude
    
    return magnitude_percent, magnitude_area

def main():
    """Main function to run the LLaMA 3 LULC event extraction pipeline."""
    args = parse_arguments()
    
    # Check if CUDA is available when device is set to cuda
    if args.device == "cuda" and not torch.cuda.is_available():
        logging.warning("CUDA requested but not available. Falling back to CPU.")
        args.device = "cpu"
    
    # Load NER output
    try:
        logging.info(f"Loading NER output from {args.input}")
        with open(args.input, 'r', encoding='utf-8') as f:
            sentences_with_entities = json.load(f)
        
        total_sentences = len(sentences_with_entities)
        logging.info(f"Loaded {total_sentences} sentences with entities")
        
        # Limit number of samples if specified
        if args.num_samples > 0 and args.num_samples < total_sentences:
            sentences_with_entities = sentences_with_entities[:args.num_samples]
            logging.info(f"Processing first {args.num_samples} sentences")
        else:
            logging.info(f"Processing all {total_sentences} sentences")
    except Exception as e:
        logging.error(f"Error loading NER output: {e}")
        return
    
    # Setup LLaMA 3 model
    try:
        model, tokenizer = setup_model(args.model, args.device, args.quantize)
    except Exception as e:
        logging.error(f"Error setting up LLaMA 3 model: {e}")
        return
    
    # Process sentences
    extracted_events = []
    
    logging.info("Starting LULC event extraction with LLaMA 3")
    for idx, entry in enumerate(tqdm(sentences_with_entities, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        # Prepare event data row
        event_data = {
            'article_id': article_id,
            'original_sentence': sentence_text,
            'llm_raw_output': 'Not Generated Yet',
            'event_found': False,
            'from_lulc': "",
            'to_lulc': "",
            'change_indicator': "",
            'lulc_process': "",
            'magnitude_percent': "",
            'magnitude_area': "",
            'error': None
        }
        
        try:
            # Construct prompt
            prompt = construct_llama3_prompt(sentence_text, entities)
            
            # Generate with LLaMA 3
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            # Parse output
            parsed_result = parse_llama3_output(generated_text)
            
            # Update event data
            event_data['event_found'] = parsed_result['event_found']
            event_data['from_lulc'] = parsed_result['from_lulc']
            event_data['to_lulc'] = parsed_result['to_lulc']
            event_data['change_indicator'] = parsed_result['change_indicator']
            event_data['lulc_process'] = parsed_result['lulc_process']
            
            # Process magnitude
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            event_data['magnitude_percent'] = magnitude_percent
            event_data['magnitude_area'] = magnitude_area
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['error'] = str(e)
            event_data['llm_raw_output'] = 'Error during generation'
        
        extracted_events.append(event_data)
        
        # Log progress periodically
        if (idx + 1) % 10 == 0:
            logging.info(f"Processed {idx + 1}/{len(sentences_with_entities)} sentences")
    
    # Create DataFrame and save results
    if extracted_events:
        results_df = pd.DataFrame(extracted_events)
        
        # Calculate statistics
        events_found = results_df['event_found'].sum()
        total_processed = len(results_df)
        event_rate = (events_found / total_processed) * 100 if total_processed > 0 else 0
        
        logging.info(f"Events found: {events_found}/{total_processed} ({event_rate:.1f}%)")
        
        # Display field completion rates for found events
        if events_found > 0:
            events_subset = results_df[results_df['event_found']]
            for field in ['from_lulc', 'to_lulc', 'change_indicator', 'lulc_process']:
                field_completion = (events_subset[field].astype(bool).sum() / events_found) * 100
                logging.info(f"  {field} completion rate: {field_completion:.1f}%")
        
        # Save to CSV
        try:
            results_df.to_csv(args.output, index=False, encoding='utf-8')
            logging.info(f"Saved results to {args.output}")
        except Exception as e:
            logging.error(f"Error saving results: {e}")
    else:
        logging.warning("No events extracted")

def run_llama3_pipeline(
    input_path=DEFAULT_NER_OUTPUT_PATH,
    output_path=DEFAULT_OUTPUT_PATH,
    model_id=DEFAULT_MODEL_ID,
    num_samples=0,
    quantize=False,
    device="cuda" if torch.cuda.is_available() else "cpu"
):
    if device == "cuda" and not torch.cuda.is_available():
        logging.warning("CUDA requested but not available. Falling back to CPU.")
        device = "cpu"

    try:
        logging.info(f"Loading NER output from {input_path}")
        with open(input_path, 'r', encoding='utf-8') as f:
            sentences_with_entities = json.load(f)

        total_sentences = len(sentences_with_entities)
        logging.info(f"Loaded {total_sentences} sentences with entities")

        if num_samples > 0 and num_samples < total_sentences:
            sentences_with_entities = sentences_with_entities[:num_samples]
            logging.info(f"Processing first {num_samples} sentences")
        else:
            logging.info(f"Processing all {total_sentences} sentences")
    except Exception as e:
        logging.error(f"Error loading NER output: {e}")
        return None

    try:
        model, tokenizer = setup_model(model_id, device, quantize)
    except Exception as e:
        logging.error(f"Error setting up LLaMA 3 model: {e}")
        return None

In [1]:
run_llama3_pipeline(num_samples=5)
print(df["llm_raw_output"].tolist())


NameError: name 'run_llama3_pipeline' is not defined

TypeError: 'NoneType' object is not subscriptable

In [24]:
# Cell 11e: LULC Change Event Extraction with helper.chat_oai (OpenAI LLM - Corrected)

import json
import pandas as pd
import logging
import re 
from tqdm import tqdm # For progress bar

# Attempt to import your custom helper module
try:
    import helper # Your custom module with chat_oai and extract_block
    HELPER_LOADED = True
    logging.info("Successfully imported 'helper' module.")
except ImportError:
    logging.error("Failed to import 'helper' module. Ensure helper.py is in the Python path or current directory.")
    logging.error("Dummy functions will be used for helper.chat_oai and helper.extract_block - LLM calls will NOT work.")
    HELPER_LOADED = False
    # Define dummy/mock functions if helper is not available
    class HelperMock:
        def chat_oai(self, system_prompt_str, user_text_str=""):
            logging.warning("HELPER_MOCK: chat_oai called. Returning dummy 'no event' JSON.")
            return """```json
{{"event_found": false, "error": "helper module not loaded correctly", "llm_raw_output": "Dummy response from HelperMock"}}
```"""
        def extract_block(self, response_text_str):
            logging.warning("HELPER_MOCK: extract_block called.")
            match = re.search(r"```json\s*\n(.*?)\n\s*```", response_text_str, re.DOTALL)
            if match: return match.group(1).strip()
            match_direct_json = re.search(r"(\{.*?\})", response_text_str, re.DOTALL) # Fallback
            if match_direct_json: return match_direct_json.group(1).strip()
            return ""
    if not HELPER_LOADED: helper = HelperMock()


# --- Configuration ---
NER_OUTPUT_JSON_PATH = "extracted_entities_structured.json" 
OAI_EVENTS_OUTPUT_PATH = "openai_extracted_lulc_events_v1.csv"
NUM_SENTENCES_TO_PROCESS = 20 

# --- 1. Load NER Output (Sentences with Entities) ---
sentences_with_entities = []
sentences_with_entities_full_list = [] 
try:
    with open(NER_OUTPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        sentences_with_entities_full_list = json.load(f)
    logging.info(f"Successfully loaded {len(sentences_with_entities_full_list)} sentences with entities from {NER_OUTPUT_JSON_PATH}")
    
    if NUM_SENTENCES_TO_PROCESS and NUM_SENTENCES_TO_PROCESS > 0 :
        if NUM_SENTENCES_TO_PROCESS < len(sentences_with_entities_full_list):
            logging.info(f"Processing first {NUM_SENTENCES_TO_PROCESS} sentences as per configuration.")
            sentences_with_entities = sentences_with_entities_full_list[:NUM_SENTENCES_TO_PROCESS]
        else: # NUM_SENTENCES_TO_PROCESS is >= available sentences
            logging.info(f"NUM_SENTENCES_TO_PROCESS ({NUM_SENTENCES_TO_PROCESS}) is >= available. Processing all {len(sentences_with_entities_full_list)}.")
            sentences_with_entities = sentences_with_entities_full_list
    elif NUM_SENTENCES_TO_PROCESS == 0 :
        logging.warning("NUM_SENTENCES_TO_PROCESS is 0. No sentences will be processed.")
        # sentences_with_entities remains empty as initialized
    else: # NUM_SENTENCES_TO_PROCESS is None or other non-positive int (like negative)
        logging.info(f"Processing all available {len(sentences_with_entities_full_list)} sentences (NUM_SENTENCES_TO_PROCESS is None or not a positive integer).")
        sentences_with_entities = sentences_with_entities_full_list

except FileNotFoundError:
    logging.error(f"ERROR: NER output file '{NER_OUTPUT_JSON_PATH}' not found. Cannot proceed.")
except json.JSONDecodeError:
    logging.error(f"ERROR: Could not decode JSON from '{NER_OUTPUT_JSON_PATH}'. File might be corrupted or not valid JSON.")
except Exception as e:
    logging.error(f"Error loading NER output JSON from '{NER_OUTPUT_JSON_PATH}': {e}", exc_info=True)

# --- 2. Define Prompt Construction and Extraction Logic ---
def format_entities_for_oai_prompt(entities_list_for_sentence):
    if not entities_list_for_sentence:
        return "No specific entities pre-identified for this sentence by prior NER."
    if not isinstance(entities_list_for_sentence, list) or not all(isinstance(e, dict) for e in entities_list_for_sentence):
        logging.warning(f"format_entities_for_oai_prompt received unexpected entity list format: {type(entities_list_for_sentence)}. Defaulting.")
        return "Entities list format error."
    formatted_str_parts = []
    for ent_dict in entities_list_for_sentence:
        text = ent_dict.get('text', 'N/A_text')
        label = ent_dict.get('label', 'N/A_label')
        formatted_str_parts.append(f"- \"{text}\" (Type: {label})")
    return "\n".join(formatted_str_parts)

def construct_oai_lulc_prompt_v1(current_sentence_text, entities_list):
    formatted_entities = format_entities_for_oai_prompt(entities_list)
    system_prompt = f"""You are an expert system tasked with extracting information about Land Use Land Cover (LULC) change events from a given sentence.
Focus on identifying a SINGLE, clear LULC change event.
An event typically involves an LULC type (e.g., forest, agriculture, urban area) undergoing a specific change or being affected by an LULC process.

The LULC types themselves are categories of land (e.g., 'forest', 'cropland', 'built-up area').
A 'change_indicator' is a word or short phrase directly describing the transformation (e.g., 'increased', 'declined', 'converted', 'loss', 'expansion').
An 'lulc_process' is a broader named process causing or constituting the change (e.g., 'urbanization', 'deforestation', 'erosion', 'afforestation'). This field might be empty.
A 'magnitude_percent' is a change described as a percentage (e.g., '15.25%').
A 'magnitude_area' is a change described with a surface area unit (e.g., '500 ha', '10 km2').

The output MUST be a JSON object enclosed in triple backticks (```json ... ```), with the following structure.
Use an empty string "" for any field if the information is not present or not applicable to the identified event.

```json
{{
  "from_lulc": "LULC_TYPE_NAME_OR_DESCRIPTION",
  "to_lulc": "LULC_TYPE_NAME_OR_DESCRIPTION",
  "change_indicator": "VERB_OR_NOUN_DESCRIBING_CHANGE",
  "lulc_process": "PROCESS_NOUN",
  "magnitude_percent": "NUMERICAL_VALUE%",
  "magnitude_area": "NUMERICAL_VALUE_WITH_AREA_UNIT"
}}```"""
    return system_prompt, current_sentence_text, formatted_entities

In [11]:
df.head()  # Show the first few rows


AttributeError: 'NoneType' object has no attribute 'head'

In [9]:
import torch
torch.device("cpu")


device(type='cpu')

In [25]:
import os
print(os.getcwd())


/home/raham/ARENA 2025


In [31]:
# Cell 12: Relation Extraction using spaCy Matcher

import json
import spacy
from spacy.matcher import Matcher # For token-based pattern matching
import pandas as pd # For potential DataFrame display later
import logging
from tqdm import tqdm

# --- Configuration ---
NER_OUTPUT_JSON_PATH = "extracted_entities_structured.json" # Input from Cell 10
MATCHER_RELATIONS_OUTPUT_JSON_PATH = "matcher_extracted_relations.jsonl" # Output as JSON Lines

# Ensure nlp_ner_pipeline from Cell 10 (with your custom EntityRuler) is loaded.
# if 'nlp_ner_pipeline' not in globals() or nlp_ner_pipeline is None:
#     logging.error("NER pipeline 'nlp_ner_pipeline' not found. Please ensure Cell 10 has run successfully.")
#     # Add logic to re-initialize nlp_ner_pipeline if necessary, using your Helper functions
#     # and BASE_SPACY_MODEL. For this example, we assume it's available.

# Global TARGET_BASE_LABELS should be defined from your config cell
# (e.g., TARGET_BASE_LABELS = ["LULC", "PROCESS", ..., "LOC"])
# Your normalize_label function (if NER output labels need normalization before matching)
# def normalize_label(label_str):
#     if label_str in ['GPE', 'NORP']: return 'LOC'
#     return label_str

# --- 1. Load NER Output (Sentences with Entities) ---
# We primarily need the sentence text. The entities are already in the Doc object when processed.
sentences_to_process_for_matcher = []
try:
    with open(NER_OUTPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        # The JSON from Cell 10 contains sentence text and pre-extracted entities
        # We'll mainly use the 'original_sentence' and 'article_id'
        all_sentence_entries = json.load(f) 
    logging.info(f"Loaded {len(all_sentence_entries)} sentence entries from {NER_OUTPUT_JSON_PATH}")
    
    # Select entries for processing (e.g., based on NUM_SENTENCES_TO_PROCESS if you have it)
    # For this example, let's assume we process all loaded entries
    sentences_to_process_for_matcher = all_sentence_entries 
    # if 'NUM_SENTENCES_TO_PROCESS' in globals() and NUM_SENTENCES_TO_PROCESS:
    #     sentences_to_process_for_matcher = all_sentence_entries[:NUM_SENTENCES_TO_PROCESS]

except FileNotFoundError: 
    logging.error(f"ERROR: NER output file '{NER_OUTPUT_JSON_PATH}' not found.")
except Exception as e: 
    logging.error(f"Error loading NER output JSON: {e}")

# --- 2. Define spaCy Matcher Patterns for Relations ---
# These patterns will operate on tokens and their attributes, including ENT_TYPE if set by your NER.

# Ensure your nlp_ner_pipeline is available
if 'nlp_ner_pipeline' not in globals() or nlp_ner_pipeline is None:
    logging.error("Critical: 'nlp_ner_pipeline' (with custom EntityRuler) is not loaded. Matcher patterns relying on custom ENT_TYPE will fail.")
    # You would need to load/recreate it here using your Helper Functions and BASE_SPACY_MODEL
    # For now, proceed assuming it might be loaded, but warn if it's just a base model later.
    if 'BASE_SPACY_MODEL' in globals():
        nlp_matcher_base = spacy.load(BASE_SPACY_MODEL) # Fallback if full pipeline not there
        logging.warning(f"Using fallback spaCy model '{BASE_SPACY_MODEL}' for Matcher. Custom ENT_TYPEs may not be present.")
    else:
        nlp_matcher_base = spacy.blank("en") # Absolute fallback
        logging.error("No spaCy model defined for Matcher. ENT_TYPE patterns will likely fail.")
    matcher = Matcher(nlp_matcher_base.vocab)
else:
    matcher = Matcher(nlp_ner_pipeline.vocab) # Use vocab from your full NER pipeline
    logging.info("Matcher initialized with vocab from 'nlp_ner_pipeline'.")


# Define relation patterns targeting your NER entity labels
# These are examples; you'll need to refine them based on your actual data and desired relations.
# Remember: ENT_TYPE here refers to the labels your nlp_ner_pipeline's EntityRuler or default NER produces.
relation_patterns = [
    { # LULC1 -> converted -> (prep) -> LULC2
        "label": "LULC_CONVERSION",
        "pattern": [
            {"ENT_TYPE": "LULC"}, # e.g., "Forest"
            {"LEMMA": {"IN": ["convert", "transition", "change", "transform"]}, "POS": "VERB"}, # e.g., "converted"
            {"LOWER": {"IN": ["to", "into"]}, "OP": "?"}, # Optional "to" or "into"
            {"ENT_TYPE": "LULC"}  # e.g., "Agriculture"
        ]
    },
    { # LULC1 -> change_verb -> (prep) -> LULC2 (More general conversion/change between two LULCs)
        "label": "LULC_TRANSFORMATION",
        "pattern": [
            {"ENT_TYPE": "LULC"},
            {"ENT_TYPE": "CHANGE", "OP":"?"}, # Optional explicit CHANGE entity
            {"POS": "VERB", "OP":"?"}, # Optional verb if CHANGE entity is a noun like "loss"
            {"POS":"ADP", "OP":"?"}, # Optional preposition like "of" or "to"
            {"ENT_TYPE": "LULC"}
        ],
        "comment": "A very broad pattern to catch two LULCs related by some verb/change term"
    },
    { # PROCESS -> causes/leads to -> CHANGE_in_LULC
        "label": "PROCESS_CAUSES_LULC_CHANGE",
        "pattern": [
            {"ENT_TYPE": "PROCESS"}, # e.g., "Urbanization"
            {"LEMMA": {"IN": ["cause", "lead", "result", "drive", "trigger"]}, "POS": "VERB", "OP": "?"},
            {"ENT_TYPE": "CHANGE", "OP": "?"}, # e.g., "loss", "increase"
            {"LOWER": "of", "OP": "?"},
            {"LOWER": "in", "OP": "?"},
            {"ENT_TYPE": "LULC"}  # e.g., "forest cover"
        ]
    },
    { # CHANGE -> in/of -> LULC -> by/of -> MAGNITUDE(PERCENT/SURFACE_UNIT)
        "label": "CHANGE_HAS_MAGNITUDE",
        "pattern": [
            {"ENT_TYPE": "CHANGE"}, # e.g., "decline", "increase"
            {"LOWER": {"IN": ["of", "in"]}, "OP": "?"},
            {"ENT_TYPE": "LULC", "OP": "?"}, # Optional LULC if clear from context of CHANGE
            {"LOWER": {"IN": ["by", "of", "to"]}, "OP": "?"}, # Preposition before magnitude
            {"ENT_TYPE": {"IN": ["PERCENT", "SURFACE_UNIT", "QUANTITY", "CARDINAL"]}} # Magnitude entity
        ]
    },
    { # LULC -> has area/is -> MAGNITUDE(SURFACE_UNIT) (Static property, not a change event)
        "label": "LULC_HAS_AREA",
        "pattern": [
            {"ENT_TYPE": "LULC"},
            {"LEMMA": {"IN": ["cover", "be", "have", "occupy"]}, "POS": "VERB", "OP": "?"}, # e.g., "covers", "is", "has an area of"
            {"LOWER": "an", "OP": "?"}, {"LOWER": "area", "OP": "?"}, {"LOWER": "of", "OP": "?"},
            {"ENT_TYPE": "SURFACE_UNIT"}
        ]
    },
    # Add more specific patterns based on your observations!
]

# Register patterns with the matcher
for pat_dict in relation_patterns:
    matcher.add(pat_dict["label"], [pat_dict["pattern"]]) # Matcher expects a list of patterns for each ID
logging.info(f"Added {len(relation_patterns)} relation patterns to the Matcher.")


# --- 3. Extract Relations using the Matcher ---
all_extracted_matcher_relations = []

if ('nlp_ner_pipeline' in globals() and nlp_ner_pipeline is not None) and sentences_to_process_for_matcher:
    logging.info(f"\n--- Applying Matcher-Based Relation Extraction on {len(sentences_to_process_for_matcher)} sentences ---")
    
    for entry in tqdm(sentences_to_process_for_matcher, desc="Matcher RE"):
        sentence_text = entry.get('original_sentence', '')
        article_id = entry.get('article_id', "Unknown")

        if not sentence_text.strip():
            continue

        # Process the sentence WITH YOUR FULL NER PIPELINE
        # This ensures doc.ents are populated correctly for the ENT_TYPE checks in patterns
        doc = nlp_ner_pipeline(sentence_text)
        
        matches = matcher(doc) # Apply the matcher to the Doc object

        for match_id, start_token_idx, end_token_idx in matches:
            relation_label = nlp_ner_pipeline.vocab.strings[match_id]  # Get the relation label string
            matched_span = doc[start_token_idx:end_token_idx] # The full span of text matched by the pattern
            
            # Extract involved entities from the matched_span more precisely
            # This requires knowing how your pattern corresponds to E1, relation_verb, E2
            # For simplicity, we'll store the full matched span and its entities.
            # A more advanced step would parse the matched_span to identify E1, E2 based on pattern structure.
            
            entities_within_match = []
            for ent in matched_span.ents: # Entities within the matched span
                 # Optionally normalize labels if needed, e.g. using your normalize_label function
                 # normalized_label = normalize_label(ent.label_)
                entities_within_match.append({"text": ent.text, "label": ent.label_})


            all_extracted_matcher_relations.append({
                "article_id": article_id,
                "sentence": sentence_text,
                "matched_text_by_pattern": matched_span.text,
                "relation_type": relation_label,
                "entities_in_match": entities_within_match # List of entities found within the matched span
            })
            
    logging.info(f"\n--- Finished Matcher-Based RE. Found {len(all_extracted_matcher_relations)} potential relations. ---")
else:
    if not ('nlp_ner_pipeline' in globals() and nlp_ner_pipeline is not None):
        logging.error("Skipping Matcher RE: 'nlp_ner_pipeline' with custom EntityRuler not available.")
    if not sentences_to_process_for_matcher:
        logging.error("Skipping Matcher RE: No sentences loaded to process.")


# --- 4. Display and Save Results ---
if all_extracted_matcher_relations:
    matcher_results_df = pd.DataFrame(all_extracted_matcher_relations)
    logging.info(f"\n--- Sample Matcher-Extracted Relations (first 20) ---")
    
    if not matcher_results_df.empty:
        # Define columns for output
        output_cols_matcher = ['article_id', 'sentence', 'relation_type', 'matched_text_by_pattern', 'entities_in_match']
        final_output_cols_matcher = [col for col in output_cols_matcher if col in matcher_results_df.columns]

        with pd.option_context('display.max_colwidth', 100, 'display.max_rows', 20):
            print(matcher_results_df[final_output_cols_matcher].head(20))
        try:
            # Saving as JSONL is good for list of dicts in 'entities_in_match'
            with open(MATCHER_RELATIONS_OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as f_out:
                for record in all_extracted_matcher_relations:
                    f_out.write(json.dumps(record) + '\n')
            logging.info(f"All matcher-extracted relations saved to: {MATCHER_RELATIONS_OUTPUT_JSONL_PATH}")
            
            # If you want a CSV, 'entities_in_match' will be a string representation of a list of dicts
            matcher_results_df[final_output_cols_matcher].to_csv("matcher_relations.csv", index=False)
        except Exception as e_save_matcher:
            logging.error(f"Error saving matcher relations: {e_save_matcher}")
    else:
        logging.info("Matcher results DataFrame is empty.")
else:
    logging.info("\nNo relations extracted using the spaCy Matcher rules.")

Matcher RE: 100%|███████████████████████████████████████████████████████████████████████████████| 1986/1986 [00:17<00:00, 111.63it/s]


    article_id  \
0    Article_2   
1    Article_2   
2    Article_2   
3    Article_2   
4    Article_2   
5    Article_4   
6    Article_4   
7    Article_4   
8    Article_4   
9    Article_7   
10   Article_7   
11   Article_7   
12  Article_10   
13  Article_11   
14  Article_11   
15  Article_12   
16  Article_12   
17  Article_12   
18  Article_12   
19  Article_12   

                                                                                               sentence  \
0   It is also defined as urban expansion, as the process of concentrating the population into citie...   
1                                  By 2019, the number of urban areas in Vietnam had increased to 846 .   
2   The percentage of agricultural and non-agricultural land in these regions' provinces has mostly ...   
3   The percentage of agricultural and non-agricultural land in these regions' provinces has mostly ...   
4   However, the conversion of land from agriculture to urban areas is concentrated in

In [32]:
# Cell 13: Convert Matcher Output to Label Studio Format (More Robust)

import json
import uuid
import re
import logging
import pandas as pd # For easily handling the matcher output if preferred

# --- Configuration ---
# Input 1: JSONL file with relations found by spaCy Matcher (output of previous Cell 12)
MATCHER_RELATIONS_FILE = "matcher_extracted_relations.jsonl"

# Input 2: JSON file with ALL NER entities for each sentence (output of Cell 10)
# This is crucial for getting accurate character offsets for ALL entities in a sentence.
FULL_NER_OUTPUT_FILE = "extracted_entities_structured.json" 

# Output file for Label Studio
LABEL_STUDIO_OUTPUT_FILE = "label_studio_tasks_with_relations.json"

# Label Studio configuration names (these MUST match your Label Studio setup)
TEXT_TAG_NAME = "text"         # The <Text name="text" ... /> tag in Label Studio
NER_LABELS_TAG_NAME = "ner"    # The <Labels name="ner" ... /> tag for NER
RELATION_LABELS_TAG_NAME = "relation" # The <RelationLabels name="relation" ... /> tag

# --- Helper Functions ---
def find_entity_in_list_by_text_and_label(entity_list, text_to_find, label_to_find=None, start_char_hint=None):
    """
    Finds an entity in a list of entity dictionaries based on text and optionally label.
    Uses start_char_hint to disambiguate if multiple entities have the same text.
    Returns the entity dictionary (with its pre-assigned 'ls_id') if found, else None.
    """
    candidates = []
    for ent in entity_list:
        if ent['text'] == text_to_find:
            if label_to_find is None or ent['label'] == label_to_find:
                candidates.append(ent)
    
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]
    
    # If multiple candidates with same text/label, use start_char_hint if available
    if start_char_hint is not None:
        for cand in candidates:
            # This requires entities_in_match from matcher to also have start/end char for disambiguation
            # For now, this part is simplified. A more robust match would use char offsets from matcher output too.
            if 'start_char' in cand and abs(cand['start_char'] - start_char_hint) < 5: # Allow small diff
                return cand
    
    logging.warning(f"Multiple entities found for text='{text_to_find}', label='{label_to_find}'. Returning first one. Disambiguation needed.")
    return candidates[0]


# --- 1. Load Full NER Output (from Cell 10) ---
# We'll put sentences and their full NER entity lists into a dictionary for easy lookup
sentence_data_map = {} # Key: (article_id, original_sentence_text), Value: list of NER entity dicts
try:
    with open(FULL_NER_OUTPUT_FILE, 'r', encoding='utf-8') as f:
        full_ner_data = json.load(f)
    for entry in full_ner_data:
        # Use a tuple of (article_id, sentence_text) as a key to handle cases where
        # the same sentence might appear under different article_ids (though less likely for LULC output)
        # Or, if sentences are unique, just sentence_text could be a key.
        # For robustness, ensure your Cell 10 output (`extracted_entities_structured.json`)
        # has unique sentence identifiers if sentence text itself can be repeated across articles.
        # Here, we assume `original_sentence` within an `article_id` is sufficiently unique.
        sentence_key = (entry.get('article_id', 'UnknownArticle'), entry.get('original_sentence', ''))
        if sentence_key[1]: # Only if sentence text is not empty
            sentence_data_map[sentence_key] = entry.get('entities', [])
    logging.info(f"Loaded {len(sentence_data_map)} unique sentences with full NER annotations from {FULL_NER_OUTPUT_FILE}")
except FileNotFoundError:
    logging.error(f"ERROR: Full NER output file '{FULL_NER_OUTPUT_FILE}' not found. This is needed for entity offsets.")
    exit() # Critical error
except Exception as e:
    logging.error(f"Error loading full NER JSON: {e}", exc_info=True)
    exit()

# --- 2. Load Matcher-Extracted Relations ---
matcher_relations_data = []
try:
    with open(MATCHER_RELATIONS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                matcher_relations_data.append(json.loads(line.strip()))
    logging.info(f"Loaded {len(matcher_relations_data)} relation candidates from {MATCHER_RELATIONS_FILE}")
except FileNotFoundError:
    logging.error(f"ERROR: Matcher relations file '{MATCHER_RELATIONS_FILE}' not found.")
    # matcher_relations_data will remain empty
except Exception as e:
    logging.error(f"Error loading matcher relations JSONL: {e}", exc_info=True)


# --- 3. Prepare Label Studio Tasks ---
label_studio_tasks = []

for rel_entry in tqdm(matcher_relations_data, desc="Preparing Label Studio Tasks"):
    article_id = rel_entry.get("article_id", "UnknownArticle")
    sentence_text = rel_entry.get("sentence", "")
    matcher_relation_type = rel_entry.get("relation_type", "UNKNOWN_RELATION")
    # Entities found *within the matcher's span* (these help identify which overall sentence entities are involved)
    entities_involved_in_match = rel_entry.get("entities_in_match", [])

    if not sentence_text:
        logging.warning(f"Skipping relation entry for {article_id} due to empty sentence.")
        continue

    # Get the full list of NER entities for this sentence using the map
    sentence_key_lookup = (article_id, sentence_text)
    full_entities_for_sentence = sentence_data_map.get(sentence_key_lookup)

    if full_entities_for_sentence is None:
        logging.warning(f"Could not find full NER data for sentence in {article_id}: '{sentence_text[:50]}...'. Skipping relation for this.")
        # Option: create task with only text if you still want to see the sentence
        # label_studio_tasks.append({"data": {"text": sentence_text}, "predictions": []})
        continue

    # Create NER pre-annotations for Label Studio, assigning unique IDs
    ls_entity_results = []
    entity_text_to_ls_id_map = {} # To map text+label to Label Studio ID for relation linking

    for ner_ent_dict in full_entities_for_sentence:
        ent_text = ner_ent_dict.get("text")
        ent_label = ner_ent_dict.get("label") # This should be your normalized label if done in Cell 10
        start_char = ner_ent_dict.get("start_char")
        end_char = ner_ent_dict.get("end_char")

        if None in [ent_text, ent_label, start_char, end_char]:
            logging.warning(f"Skipping malformed NER entity in {article_id} for sentence '{sentence_text[:50]}...': {ner_ent_dict}")
            continue
        
        ls_ent_id = str(uuid.uuid4()) # Generate unique ID for Label Studio
        ls_entity_results.append({
            "id": ls_ent_id,
            "from_name": NER_LABELS_TAG_NAME, # Must match your <Labels name="ner" ...>
            "to_name": TEXT_TAG_NAME,         # Must match your <Text name="text" ...>
            "type": "labels",
            "value": {
                "start": start_char,
                "end": end_char,
                "text": ent_text,
                "labels": [ent_label] # Label Studio expects a list of labels
            }
        })
        # Store mapping: use a tuple of (text, label, start_char) for more uniqueness if needed
        # For simplicity, if text is unique enough within a sentence for a given role:
        entity_text_to_ls_id_map[(ent_text, ent_label)] = ls_ent_id # Simpler map

    # Now, create relation pre-annotations based on matcher_relation_type and entities_involved_in_match
    # This is the part that needs to be specific to your rules' semantics.
    ls_relation_results = []
    
    # Crude logic: assume first two LULC entities in entities_involved_in_match form the relation
    # YOU NEED TO REFINE THIS BASED ON YOUR MATCHER PATTERN SEMANTICS FOR EACH relation_type
    
    # Example: For LULC_CONVERSION ("LULC1 converted to LULC2")
    if matcher_relation_type == "LULC_CONVERSION" and len(entities_involved_in_match) >= 2:
        # Try to find the LULC entities that your pattern intended to be E1 (from) and E2 (to)
        # This logic assumes the order in 'entities_in_match' reflects the pattern order,
        # or that the relevant entities have distinct labels we can pick out.
        
        e1_text, e1_label, e2_text, e2_label = None, None, None, None
        
        # Find LULC entities involved in the match
        matched_lulc_entities = [e for e in entities_involved_in_match if e.get("label") == "LULC"]
        
        if len(matched_lulc_entities) >= 2:
            e1_text = matched_lulc_entities[0].get("text")
            e1_label = matched_lulc_entities[0].get("label") # Should be "LULC"
            e2_text = matched_lulc_entities[1].get("text")
            e2_label = matched_lulc_entities[1].get("label") # Should be "LULC"

            # Find their LS IDs from the full sentence NER list
            from_ent_ls_id = entity_text_to_ls_id_map.get((e1_text, e1_label))
            to_ent_ls_id = entity_text_to_ls_id_map.get((e2_text, e2_label))

            if from_ent_ls_id and to_ent_ls_id and from_ent_ls_id != to_ent_ls_id:
                ls_relation_results.append({
                    "from_id": from_ent_ls_id,
                    "to_id": to_ent_ls_id,
                    "type": "relation",
                    "direction": "right", # Or based on your relation type
                    "labels": [matcher_relation_type] # Use the matcher's relation type as the label
                })
            else:
                logging.warning(f"Could not find LS IDs for LULC_CONVERSION entities in '{sentence_text[:50]}...' "
                                f"E1: ('{e1_text}', '{e1_label}'), E2: ('{e2_text}', '{e2_label}')")
    
    # TODO: Add specific logic for other relation_types from your Matcher
    # For example, for CHANGE_HAS_MAGNITUDE:
    # elif matcher_relation_type == "CHANGE_HAS_MAGNITUDE":
    #     change_ent = find_entity_in_list_by_text_and_label(entities_involved_in_match, "CHANGE_TEXT_HINT", "CHANGE")
    #     magnitude_ent = find_entity_in_list_by_text_and_label(entities_involved_in_match, "MAGNITUDE_TEXT_HINT", "PERCENT") # or SURFACE_UNIT
    #     if change_ent and magnitude_ent:
    #         from_id = entity_text_to_ls_id_map.get((change_ent['text'], change_ent['label']))
    #         to_id = entity_text_to_ls_id_map.get((magnitude_ent['text'], magnitude_ent['label']))
    #         if from_id and to_id:
    #             ls_relation_results.append({ ... "labels": ["has_magnitude"] ... })


    # Combine NER results and Relation results for Label Studio "result" field
    combined_results_for_ls = ls_entity_results + ls_relation_results # Order might matter for LS UI
    
    if combined_results_for_ls: # Only create task if there's something to annotate (at least NER)
        label_studio_tasks.append({
            "data": {
                # You can add article_id here too if you configure LS to display it
                "text": sentence_text, 
                "article_id_ls": article_id # Custom data field for LS
            },
            "predictions": [{ # Using "predictions" key for pre-annotations
                "model_version": "rule_matcher_v1", # Optional: version your rules
                "score": 0.9, # Optional: a dummy confidence score for the prediction
                "result": combined_results_for_ls
            }]
        })

# --- 4. Save Label Studio Formatted Data ---
try:
    with open(LABEL_STUDIO_OUTPUT_FILE, "w", encoding="utf-8") as f_out:
        json.dump(label_studio_tasks, f_out, indent=2, ensure_ascii=False)
    logging.info(f"✅ Label Studio pre-annotation file saved to {LABEL_STUDIO_OUTPUT_FILE} with {len(label_studio_tasks)} tasks.")
except Exception as e_save_ls:
    logging.error(f"Error saving Label Studio JSON: {e_save_ls}", exc_info=True)

Preparing Label Studio Tasks: 100%|█████████████████████████████████████████████████████████████| 363/363 [00:00<00:00, 18503.16it/s]


In [54]:
import os
print(os.path.exists("matcher_extracted_relations.jsonl"))
# Or, if you used a different path/name:
# print(os.path.exists(MATCHER_RELATIONS_OUTPUT_JSONL_PATH)) # Use the variable from Cell 12

False


In [37]:
# Cell 13: Convert NER and Matcher Relations to Label Studio Format

import json
import uuid # For generating unique IDs for Label Studio entities
import logging
from tqdm import tqdm # For progress bar

# --- Configuration ---
# Input 1: JSON file with ALL NER entities for each sentence (output of Cell 10)
FULL_NER_OUTPUT_FILE = "extracted_entities_structured.json" 

# Input 2: JSONL file with relations found by spaCy Matcher (output of Cell 12)
MATCHER_RELATIONS_FILE = "matcher_extracted_relations.jsonl" 

# Output file for Label Studio
LABEL_STUDIO_OUTPUT_FILE = "label_studio_tasks_with_ner_and_relations.json"

# Label Studio configuration names (MUST match your Label Studio XML setup)
TEXT_TAG_NAME = "text"         # The <Text name="text" ... /> tag
NER_LABELS_TAG_NAME = "ner"    # The <Labels name="ner" ... /> tag for NER
RELATION_LABELS_TAG_NAME = "relation" # The <RelationLabels name="relation" ... /> tag for Relations
# (You'll define actual relation labels like "converted_to", "has_magnitude" inside the <RelationLabels> tag in LS)


# --- 1. Load Full NER Output (from Cell 10) into a more usable structure ---
# Map: (article_id, original_sentence_text) -> list of NER entity dicts from Cell 10
sentence_to_full_ner_map = {}
try:
    with open(FULL_NER_OUTPUT_FILE, 'r', encoding='utf-8') as f:
        full_ner_data_from_cell10 = json.load(f)
    for entry in full_ner_data_from_cell10:
        sentence_key = (entry.get('article_id', 'UnknownArticle'), entry.get('original_sentence', ''))
        if sentence_key[1]: # Only if sentence text is not empty
            sentence_to_full_ner_map[sentence_key] = entry.get('entities', []) # List of NER entity dicts
    logging.info(f"Loaded {len(sentence_to_full_ner_map)} unique sentences with full NER from {FULL_NER_OUTPUT_FILE}")
except FileNotFoundError:
    logging.error(f"CRITICAL ERROR: Full NER output file '{FULL_NER_OUTPUT_FILE}' not found. This file is essential.")
    exit() # Stop execution if this crucial file is missing
except Exception as e:
    logging.error(f"Error loading full NER JSON from '{FULL_NER_OUTPUT_FILE}': {e}", exc_info=True)
    exit()

# --- 2. Load Matcher-Extracted Relations (from Cell 12) ---
# Group matcher relations by sentence for easier lookup
matcher_relations_by_sentence = {} # Key: (article_id, sentence_text), Value: list of matcher_relation_dicts
try:
    with open(MATCHER_RELATIONS_FILE, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            if line.strip():
                try:
                    rel_entry = json.loads(line.strip())
                    sentence_key = (rel_entry.get("article_id", "UnknownArticle"), rel_entry.get("sentence", ""))
                    if sentence_key[1]: # If sentence text is not empty
                        if sentence_key not in matcher_relations_by_sentence:
                            matcher_relations_by_sentence[sentence_key] = []
                        matcher_relations_by_sentence[sentence_key].append(rel_entry)
                except json.JSONDecodeError:
                    logging.warning(f"Skipping malformed JSON line {line_idx+1} in {MATCHER_RELATIONS_FILE}")
    logging.info(f"Loaded relations for {len(matcher_relations_by_sentence)} unique sentences from {MATCHER_RELATIONS_FILE}")
except FileNotFoundError:
    logging.warning(f"Matcher relations file '{MATCHER_RELATIONS_FILE}' not found. No relation pre-annotations will be added.")
    # matcher_relations_by_sentence will remain empty
except Exception as e:
    logging.error(f"Error loading matcher relations JSONL from '{MATCHER_RELATIONS_FILE}': {e}", exc_info=True)


# --- 3. Prepare Label Studio Tasks ---
label_studio_tasks = []

# We iterate through the sentences that have full NER annotations.
# For each of these, we'll check if the matcher found any relations.
logging.info(f"Preparing Label Studio tasks for {len(sentence_to_full_ner_map)} sentences...")
for sentence_key, full_entities_for_this_sentence in tqdm(sentence_to_full_ner_map.items(), desc="Creating Label Studio Tasks"):
    article_id, sentence_text = sentence_key

    ls_entity_results_for_sentence = [] # For Label Studio NER <Labels>
    # This map will help find the Label Studio 'id' of an entity based on its properties from NER
    # Key: (text, label, start_char), Value: ls_entity_id
    entity_signature_to_ls_id_map = {} 

    # Create NER pre-annotations for this sentence
    for ner_ent_dict in full_entities_for_this_sentence:
        ent_text = ner_ent_dict.get("text")
        ent_label = ner_ent_dict.get("label") 
        start_char = ner_ent_dict.get("start_char")
        end_char = ner_ent_dict.get("end_char")

        if None in [ent_text, ent_label] or start_char is None or end_char is None: # Check for Nones
            logging.warning(f"Skipping malformed NER entity in {article_id} for sentence '{sentence_text[:50]}...': {ner_ent_dict}")
            continue
        
        ls_ent_id = str(uuid.uuid4()) 
        ls_entity_results_for_sentence.append({
            "id": ls_ent_id,
            "from_name": NER_LABELS_TAG_NAME, 
            "to_name": TEXT_TAG_NAME,       
            "type": "labels",
            "value": {
                "start": start_char, "end": end_char, "text": ent_text,
                "labels": [ent_label] # Label Studio expects a list of labels
            }
        })
        # More robust key for mapping: (start_char, end_char, label) - less prone to text collision
        entity_signature_to_ls_id_map[(start_char, end_char, ent_label)] = ls_ent_id

    ls_relation_results_for_sentence = [] # For Label Studio <Relations>

    # Check if the matcher found any relations for this specific sentence
    matched_relations_for_this_sentence = matcher_relations_by_sentence.get(sentence_key, [])

    for matcher_rel_entry in matched_relations_for_this_sentence:
        matcher_relation_type = matcher_rel_entry.get("relation_type", "UNKNOWN_MATCHER_REL")
        # Entities identified by NER *within the span matched by the specific rule*
        entities_involved_in_rule_match = matcher_rel_entry.get("entities_in_match", [])
        
        # --- LOGIC TO IDENTIFY from_entity and to_entity for the relation ---
        # This part is CRITICAL and needs to be specific to each matcher_relation_type
        from_entity_ls_id = None
        to_entity_ls_id = None
        # This will be the label you assign in Label Studio for the relation
        actual_ls_relation_label = None 

        if matcher_relation_type == "LULC_CONVERSION":
            # Assumption: For LULC_CONVERSION, 'entities_in_match' will contain at least two "LULC" entities,
            # and their order implies from -> to based on your Matcher pattern.
            lulc_entities_in_matcher_span = [e for e in entities_involved_in_rule_match if e.get("label") == "LULC"]
            if len(lulc_entities_in_matcher_span) >= 2:
                # Get start_char for disambiguation. These start_chars are from the Matcher output.
                e1_start = lulc_entities_in_matcher_span[0].get("start_char_in_sentence")
                e2_start = lulc_entities_in_matcher_span[1].get("start_char_in_sentence")
                
                from_entity_ls_id = entity_signature_to_ls_id_map.get((e1_start, lulc_entities_in_matcher_span[0].get("end_char_in_sentence"), "LULC"))
                to_entity_ls_id   = entity_signature_to_ls_id_map.get((e2_start, lulc_entities_in_matcher_span[1].get("end_char_in_sentence"), "LULC"))
                actual_ls_relation_label = "converted_to" # Example Label Studio relation label
        
        elif matcher_relation_type == "CHANGE_HAS_MAGNITUDE_PERCENT":
            change_ent_dict = next((e for e in entities_involved_in_rule_match if e.get("label") == "CHANGE"), None)
            percent_ent_dict = next((e for e in entities_involved_in_rule_match if e.get("label") == "PERCENT"), None)
            if change_ent_dict and percent_ent_dict:
                from_entity_ls_id = entity_signature_to_ls_id_map.get((change_ent_dict.get("start_char_in_sentence"), change_ent_dict.get("end_char_in_sentence"), "CHANGE"))
                to_entity_ls_id   = entity_signature_to_ls_id_map.get((percent_ent_dict.get("start_char_in_sentence"), percent_ent_dict.get("end_char_in_sentence"), "PERCENT"))
                actual_ls_relation_label = "has_magnitude"
        
        elif matcher_relation_type == "CHANGE_HAS_MAGNITUDE_AREA":
            change_ent_dict = next((e for e in entities_involved_in_rule_match if e.get("label") == "CHANGE"), None)
            area_ent_dict = next((e for e in entities_involved_in_rule_match if e.get("label") == "SURFACE_UNIT"), None)
            if change_ent_dict and area_ent_dict:
                from_entity_ls_id = entity_signature_to_ls_id_map.get((change_ent_dict.get("start_char_in_sentence"), change_ent_dict.get("end_char_in_sentence"), "CHANGE"))
                to_entity_ls_id   = entity_signature_to_ls_id_map.get((area_ent_dict.get("start_char_in_sentence"), area_ent_dict.get("end_char_in_sentence"), "SURFACE_UNIT"))
                actual_ls_relation_label = "has_magnitude"

        # Add more 'elif matcher_relation_type == "YOUR_OTHER_TYPE":' blocks here
        # For each, determine how to pick the from_entity and to_entity from 'entities_involved_in_rule_match'
        # and what 'actual_ls_relation_label' to use.

        if from_entity_ls_id and to_entity_ls_id and actual_ls_relation_label and from_entity_ls_id != to_entity_ls_id:
            ls_relation_results_for_sentence.append({
                "from_id": from_entity_ls_id,
                "to_id": to_entity_ls_id,
                "type": "relation",
                "direction": "right", # Default, adjust if your relation is directional
                "labels": [actual_ls_relation_label] 
            })
        elif entities_involved_in_rule_match and actual_ls_relation_label is None : # A match was found, but logic to make LS relation failed
            logging.warning(f"Matcher found '{matcher_relation_type}' for article '{article_id}' but could not map it to a Label Studio relation. "
                            f"Matched text: '{matcher_rel_entry.get('matched_text_by_pattern', '')[:50]}...'")


    # Combine all NER entities and successfully created relations for this sentence
    final_ls_results_for_this_task = ls_entity_results_for_sentence + ls_relation_results_for_sentence
    
    if final_ls_results_for_this_task: # Only add task if there's something to show (at least NER)
        label_studio_tasks.append({
            "data": {
                TEXT_TAG_NAME: sentence_text, # Use variable for text tag name
                "article_id_ls": article_id  # Optional: custom data field for article ID
            },
            "predictions": [{ # Using "predictions" for pre-annotations
                "model_version": "ner_v1_plus_matcher_v1", 
                "score": 0.9, # Dummy score
                "result": final_ls_results_for_this_task
            }]
        })

# --- 4. Save Label Studio Formatted Data ---
try:
    with open(LABEL_STUDIO_OUTPUT_FILE, "w", encoding="utf-8") as f_out:
        json.dump(label_studio_tasks, f_out, indent=2, ensure_ascii=False)
    logging.info(f"✅ Label Studio pre-annotation file saved to {LABEL_STUDIO_OUTPUT_FILE} with {len(label_studio_tasks)} tasks.")
except Exception as e_save_ls:
    logging.error(f"Error saving Label Studio JSON to '{LABEL_STUDIO_OUTPUT_FILE}': {e_save_ls}", exc_info=True)

Creating Label Studio Tasks:   0%|                                                                          | 0/1950 [00:00<?, ?it/s]WARNING:root:Matcher found 'CHANGE_HAS_MAGNITUDE' for article 'Article_2' but could not map it to a Label Studio relation. Matched text: 'increase to about...'


Creating Label Studio Tasks:  42%|██████████████████████████▏                                   | 823/1950 [00:00<00:00, 8209.16it/s]WARNING:root:Matcher found 'CHANGE_HAS_MAGNITUDE' for article 'Article_50' but could not map it to a Label Studio relation. Matched text: 'decreased to 4,571...'


Creating Label Studio Tasks:  84%|███████████████████████████████████████████████████▍         | 1644/1950 [00:00<00:00, 6305.21it/s]WARNING:root:Matcher found 'CHANGE_HAS_MAGNITUDE' for article 'Article_98' but could not map it to a Label Studio relation. Matched text: 'decrease of 0.25...'


Creating Label Studio Tasks: 100%|█████████████████████████████████████████████████████████████| 1950/1950 [00:00<00:00, 6436.70it/s]


In [25]:
"""
LLaMA 3 Implementation for LULC Event Extraction - Jupyter Notebook Version
===========================================================================

This notebook demonstrates how to use LLaMA 3 for extracting Land Use Land Cover (LULC) 
change events from text in a Jupyter notebook environment.

Requirements:
- Python 3.8+
- PyTorch 2.0+
- transformers 4.30.0+
- GPU with at least 16GB VRAM (for 8B model)
"""

# Cell 1: Import libraries and configure logging
import json
import os
import re
import logging
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)

# Cell 2: Configuration settings
# You can modify these settings as needed
NER_OUTPUT_PATH = "extracted_entities_structured.json"
OUTPUT_PATH = "llama3_extracted_lulc_events.csv"
MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # 8B version
# Alternative models:
# "meta-llama/Meta-Llama-3-8B-Instruct" - Instruction-tuned version
# "meta-llama/Meta-Llama-3-70B" - Larger model (requires more VRAM)
NUM_SAMPLES = 0  # 0 for all samples, or specify a number to limit
USE_QUANTIZATION = True  # Set to True to use 4-bit quantization (reduces memory usage)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Cell 3: Setup model and tokenizer
def setup_model(model_id, device, use_quantization=False):
    """
    Load the LLaMA 3 model and tokenizer.
    
    Args:
        model_id: Hugging Face model ID
        device: Device to load the model on
        use_quantization: Whether to use 4-bit quantization
        
    Returns:
        model, tokenizer
    """
    logging.info(f"Loading LLaMA 3 tokenizer: {model_id}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Configure quantization if requested
    if use_quantization:
        logging.info("Using 4-bit quantization")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    else:
        quantization_config = None
    
    # Load model with appropriate configuration
    logging.info(f"Loading LLaMA 3 model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device,
        quantization_config=quantization_config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    
    logging.info(f"LLaMA 3 model loaded successfully on {device}")
    return model, tokenizer

# Execute this cell to load the model
model, tokenizer = setup_model(MODEL_ID, DEVICE, USE_QUANTIZATION)

# Cell 4: Prompt construction functions
def format_entities_for_prompt(entities):
    """Format entities for inclusion in the prompt."""
    if not entities:
        return "No specific entities pre-identified."
    
    # Group entities by type
    entities_by_type = {}
    for entity in entities:
        entity_type = entity.get('label', 'UNKNOWN')
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity.get('text', 'N/A'))
    
    # Format grouped entities
    formatted_lines = []
    for entity_type, entity_texts in entities_by_type.items():
        unique_texts = list(set(entity_texts))  # Remove duplicates
        # Fixed line to avoid nested f-string syntax error
        entity_list = ", ".join(['"' + text + '"' for text in unique_texts])
        formatted_lines.append(f"- {entity_type}: {entity_list}")
    
    return "\n".join(formatted_lines)

def construct_llama3_prompt(sentence_text, entities):
    """
    Construct an optimized prompt for LLaMA 3.
    
    This prompt is specifically designed for LLaMA 3's capabilities and includes:
    1. Clear task definition
    2. Structured format instructions
    3. Few-shot examples
    4. Entity context
    """
    formatted_entities = format_entities_for_prompt(entities)
    
    # Extract LULC entities specifically for better context
    lulc_entities = [ent for ent in entities if ent.get('label') == 'LULC']
    # Fixed line to avoid nested f-string syntax error
    lulc_text = ", ".join(['"' + ent.get("text") + '"' for ent in lulc_entities]) if lulc_entities else "None identified"
    
    # Extract change indicators for better context
    change_entities = [ent for ent in entities if ent.get('label') == 'CHANGE']
    # Fixed line to avoid nested f-string syntax error
    change_text = ", ".join(['"' + ent.get("text") + '"' for ent in change_entities]) if change_entities else "None identified"
    
    prompt = f"""<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
</context>

<input>
"{sentence_text}"
</input>

<entities>
{formatted_entities}
</entities>

<instructions>
Analyze the input text and extract a LULC change event with these components:

FROM: The original land use/cover type that is changing or being converted
TO: The resulting land use/cover type
CHANGE: Words indicating change (e.g., increase, decrease, conversion)
PROCESS: The broader process (e.g., deforestation, urbanization)
MAGNITUDE: Any percentage or area measurement

If no LULC change event is present, respond with "NO_EVENT".
</instructions>

<examples>
Example 1:
Input: "Forest cover declined by 15% in the region."
Entities:
- LULC: "Forest"
- CHANGE: "declined"
- PERCENT: "15%"
- LOC: "region"

Output:
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%

Example 2:
Input: "Agricultural land was converted to urban areas between 2010 and 2020."
Entities:
- LULC: "Agricultural land", "urban areas"
- CHANGE: "converted"
- DATE: "2010", "2020"

Output:
FROM: agricultural land
TO: urban areas
CHANGE: converted
PROCESS: urbanization
MAGNITUDE: 

Example 3:
Input: "The study examined biodiversity in tropical forests."
Entities:
- LULC: "tropical forests"

Output:
NO_EVENT

Example 4:
Input: "Built-up area increased from 52.88% in 2002 to 65.5% in 2018, a change of 12.77%."
Entities:
- LULC: "Built-up area"
- CHANGE: "increased"
- PERCENT: "52.88%", "65.5%", "12.77%"
- DATE: "2002", "2018"

Output:
FROM: built-up area
TO: built-up area
CHANGE: increased
PROCESS: urbanization
MAGNITUDE: 12.77%
</examples>

<output>
"""
    return prompt

# Cell 5: Text generation and parsing functions
def generate_with_llama3(model, tokenizer, prompt, max_new_tokens=256):
    """
    Generate text using LLaMA 3 model.
    
    Args:
        model: LLaMA 3 model
        tokenizer: LLaMA 3 tokenizer
        prompt: Input prompt
        max_new_tokens: Maximum number of tokens to generate
        
    Returns:
        Generated text
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate with appropriate parameters for structured extraction
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,  # Low temperature for deterministic outputs
            top_p=0.9,
            do_sample=True,  # Light sampling for better quality
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_output[len(tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)):]
    
    # Clean up the output
    generated_text = generated_text.strip()
    
    # If the output contains </output> tag, extract only the content before it
    if "</output>" in generated_text:
        generated_text = generated_text.split("</output>")[0].strip()
    
    return generated_text

def parse_llama3_output(output_text):
    """
    Parse the LLaMA 3 output into structured fields.
    
    Args:
        output_text: Raw output from LLaMA 3
        
    Returns:
        Dictionary with parsed fields
    """
    # Check for NO_EVENT marker
    if "NO_EVENT" in output_text:
        return {
            "event_found": False,
            "from_lulc": "",
            "to_lulc": "",
            "change_indicator": "",
            "lulc_process": "",
            "magnitude": ""
        }
    
    # Extract fields using regex
    from_match = re.search(r'FROM:\s*(.*?)(?=\nTO:|$)', output_text, re.DOTALL)
    to_match = re.search(r'TO:\s*(.*?)(?=\nCHANGE:|$)', output_text, re.DOTALL)
    change_match = re.search(r'CHANGE:\s*(.*?)(?=\nPROCESS:|$)', output_text, re.DOTALL)
    process_match = re.search(r'PROCESS:\s*(.*?)(?=\nMAGNITUDE:|$)', output_text, re.DOTALL)
    magnitude_match = re.search(r'MAGNITUDE:\s*(.*?)(?=\n|$)', output_text, re.DOTALL)
    
    # Extract values or default to empty string
    from_lulc = from_match.group(1).strip() if from_match else ""
    to_lulc = to_match.group(1).strip() if to_match else ""
    change_indicator = change_match.group(1).strip() if change_match else ""
    lulc_process = process_match.group(1).strip() if process_match else ""
    magnitude = magnitude_match.group(1).strip() if magnitude_match else ""
    
    # Determine if an event was found (at least one field has content)
    event_found = bool(from_lulc or to_lulc or change_indicator or lulc_process)
    
    return {
        "event_found": event_found,
        "from_lulc": from_lulc,
        "to_lulc": to_lulc,
        "change_indicator": change_indicator,
        "lulc_process": lulc_process,
        "magnitude": magnitude
    }

def process_magnitude(magnitude):
    """
    Process magnitude into percent and area components.
    
    Args:
        magnitude: Raw magnitude string
        
    Returns:
        Tuple of (magnitude_percent, magnitude_area)
    """
    if not magnitude:
        return "", ""
    
    magnitude_percent = ""
    magnitude_area = ""
    
    # Check for percentage
    if "%" in magnitude:
        magnitude_percent = magnitude
    # Check for area units
    elif any(unit in magnitude.lower() for unit in ["ha", "km", "acre", "meter", "sq", "hectare"]):
        magnitude_area = magnitude
    # Check for numbers with area units using regex
    elif re.search(r'\d+\s*(?:ha|km|m|acre)', magnitude, re.IGNORECASE):
        magnitude_area = magnitude
    # If it's just a number, try to determine if it's a percentage
    elif re.search(r'\d+\.\d+|\d+', magnitude):
        try:
            value = float(re.search(r'\d+\.\d+|\d+', magnitude).group())
            if value <= 100:
                magnitude_percent = magnitude
            else:
                magnitude_area = magnitude
        except:
            magnitude_area = magnitude
    else:
        magnitude_area = magnitude
    
    return magnitude_percent, magnitude_area

# Cell 6: Load NER output data
# Execute this cell to load your data
try:
    logging.info(f"Loading NER output from {NER_OUTPUT_PATH}")
    with open(NER_OUTPUT_PATH, 'r', encoding='utf-8') as f:
        sentences_with_entities = json.load(f)
    
    total_sentences = len(sentences_with_entities)
    logging.info(f"Loaded {total_sentences} sentences with entities")
    
    # Limit number of samples if specified
    if NUM_SAMPLES > 0 and NUM_SAMPLES < total_sentences:
        sentences_with_entities = sentences_with_entities[:NUM_SAMPLES]
        logging.info(f"Processing first {NUM_SAMPLES} sentences")
    else:
        logging.info(f"Processing all {total_sentences} sentences")
        
    # Display first example
    print("\nFirst example:")
    print(f"Sentence: {sentences_with_entities[0].get('original_sentence', '')}")
    print("Entities:")
    for entity in sentences_with_entities[0].get('entities', []):
        print(f"  - {entity.get('text', '')} ({entity.get('label', '')})")
        
except Exception as e:
    logging.error(f"Error loading NER output: {e}")
    sentences_with_entities = []

# Cell 7: Process a single example (for testing)
# Execute this cell to test extraction on a single example
if sentences_with_entities:
    # Get the first example
    example = sentences_with_entities[0]
    sentence_text = example.get('original_sentence', '')
    entities = example.get('entities', [])
    
    # Construct prompt
    prompt = construct_llama3_prompt(sentence_text, entities)
    print("Prompt:")
    print(prompt)
    
    # Generate with LLaMA 3
    generated_text = generate_with_llama3(model, tokenizer, prompt)
    print("\nGenerated text:")
    print(generated_text)
    
    # Parse output
    parsed_result = parse_llama3_output(generated_text)
    print("\nParsed result:")
    for key, value in parsed_result.items():
        print(f"  {key}: {value}")
    
    # Process magnitude
    if parsed_result['magnitude']:
        magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
        print(f"  magnitude_percent: {magnitude_percent}")
        print(f"  magnitude_area: {magnitude_area}")

# Cell 8: Process all examples
# Execute this cell to process all examples
def process_all_examples():
    extracted_events = []
    
    logging.info("Starting LULC event extraction with LLaMA 3")
    for idx, entry in enumerate(tqdm(sentences_with_entities, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        # Prepare event data row
        event_data = {
            'article_id': article_id,
            'original_sentence': sentence_text,
            'llm_raw_output': 'Not Generated Yet',
            'event_found': False,
            'from_lulc': "",
            'to_lulc': "",
            'change_indicator': "",
            'lulc_process': "",
            'magnitude_percent': "",
            'magnitude_area': "",
            'error': None
        }
        
        try:
            # Construct prompt
            prompt = construct_llama3_prompt(sentence_text, entities)
            
            # Generate with LLaMA 3
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            # Parse output
            parsed_result = parse_llama3_output(generated_text)
            
            # Update event data
            event_data['event_found'] = parsed_result['event_found']
            event_data['from_lulc'] = parsed_result['from_lulc']
            event_data['to_lulc'] = parsed_result['to_lulc']
            event_data['change_indicator'] = parsed_result['change_indicator']
            event_data['lulc_process'] = parsed_result['lulc_process']
            
            # Process magnitude
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            event_data['magnitude_percent'] = magnitude_percent
            event_data['magnitude_area'] = magnitude_area
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['error'] = str(e)
            event_data['llm_raw_output'] = 'Error during generation'
        
        extracted_events.append(event_data)
    
    return extracted_events



2025-05-21 17:09:31,887 - INFO - Loading LLaMA 3 tokenizer: meta-llama/Meta-Llama-3-8B
2025-05-21 17:09:32,515 - INFO - Using 4-bit quantization
2025-05-21 17:09:32,519 - INFO - Loading LLaMA 3 model: meta-llama/Meta-Llama-3-8B


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-05-21 17:09:42,093 - INFO - LLaMA 3 model loaded successfully on cuda
2025-05-21 17:09:42,148 - INFO - Loading NER output from extracted_entities_structured.json
2025-05-21 17:09:42,171 - INFO - Loaded 1986 sentences with entities
2025-05-21 17:09:42,172 - INFO - Processing all 1986 sentences



First example:
Sentence: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.
Entities:
  - results (CHANGE)
  - Thimphu (LOC)
  - city (LULC)
  - changed (CHANGE)
  - change (CHANGE)
  - 2050 (DATE)
Prompt:
<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
</context>

<input>
"Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050."
</input>

<entities>
- CHANGE: "changed", "results", "chan

In [27]:
# Cell 9: Analyze and save results
# Execute this cell after processing all examples
def analyze_and_save_results(extracted_events):
    if not extracted_events:
        logging.warning("No events extracted")
        return None
    
    # Create DataFrame
    results_df = pd.DataFrame(extracted_events)
    
    # Calculate statistics
    events_found = results_df['event_found'].sum()
    total_processed = len(results_df)
    event_rate = (events_found / total_processed) * 100 if total_processed > 0 else 0
    
    logging.info(f"Events found: {events_found}/{total_processed} ({event_rate:.1f}%)")
    
    # Display field completion rates for found events
    if events_found > 0:
        events_subset = results_df[results_df['event_found']]
        for field in ['from_lulc', 'to_lulc', 'change_indicator', 'lulc_process']:
            field_completion = (events_subset[field].astype(bool).sum() / events_found) * 100
            logging.info(f"  {field} completion rate: {field_completion:.1f}%")
    
    # Save to CSV
    try:
        results_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
        logging.info(f"Saved results to {OUTPUT_PATH}")
    except Exception as e:
        logging.error(f"Error saving results: {e}")
    
    return results_df

# Uncomment and run this cell after processing all examples
# results_df = analyze_and_save_results(extracted_events)


In [29]:
# Display first 10 results
if 'results_df' in globals() and not results_df.empty:
    display_cols = ['article_id', 'original_sentence', 'event_found', 
                    'from_lulc', 'to_lulc', 'change_indicator', 'lulc_process',
                    'magnitude_percent', 'magnitude_area']
    display(results_df[display_cols].head(10))
else:
    print("Results not available. Make sure to run the processing and analysis cells first.")


Results not available. Make sure to run the processing and analysis cells first.


In [2]:
extracted_events = process_all_examples()


NameError: name 'process_all_examples' is not defined

In [26]:
results_df = analyze_and_save_results(extracted_events)


NameError: name 'analyze_and_save_results' is not defined